In [1]:
import pandas as pd
from selenium import webdriver
import json
import numpy as np
from bs4 import BeautifulSoup
import time
import requests
from tqdm import tqdm

In [2]:
def extract_5top_league_fotmob():
    response = requests.get("https://www.fotmob.com/api/data/allLeagues?locale=en&country=ESP")
    response.raise_for_status()
    data = response.json()
    df_countries = pd.DataFrame(data['countries'])

    df_exploded = df_countries.explode('leagues')
    df_leagues = pd.json_normalize(df_exploded['leagues'])
    df_leagues = df_leagues.add_prefix('league_')

    df_final = df_exploded.drop(columns=['leagues']).reset_index(drop=True).join(df_leagues)
    df_final= df_final[['ccode', 'name'  ,'league_id', 'league_name', 'league_pageUrl' ]].rename(columns={'ccode': 'ccode_country', 'name': 'name_country'})
    df_final['league_pageUrl'] = 'https://www.fotmob.com' + df_final['league_pageUrl'] 
    df_final['league_name_display'] = df_final['league_name'] + ' - ' + df_final['name_country']   
    df_final['league_logo_url'] = 'https://images.fotmob.com/image_resources/logo/leaguelogo/' + df_final['league_id'].astype(str) + '.png'
    top5 = [
        "Premier League - England",
        "LaLiga - Spain",
        "Serie A - Italy",
        "Bundesliga - Germany",
        "Ligue 1 - France"
    ]

    df_top5 = df_final[df_final["league_name_display"].isin(top5)].reset_index(drop=True)
    return df_top5      

In [3]:
df_top5= extract_5top_league_fotmob()
df_top5

,ccode_country,name_country,league_id,league_name,league_pageUrl,league_name_display,league_logo_url
0,ENG,England,47,Premier League,https://www.fotmob.com/leagues/47/overview/pre...,Premier League - England,https://images.fotmob.com/image_resources/logo...
1,FRA,France,53,Ligue 1,https://www.fotmob.com/leagues/53/overview/lig...,Ligue 1 - France,https://images.fotmob.com/image_resources/logo...
2,GER,Germany,54,Bundesliga,https://www.fotmob.com/leagues/54/overview/bun...,Bundesliga - Germany,https://images.fotmob.com/image_resources/logo...
3,ITA,Italy,55,Serie A,https://www.fotmob.com/leagues/55/overview/serie,Serie A - Italy,https://images.fotmob.com/image_resources/logo...
4,ESP,Spain,87,LaLiga,https://www.fotmob.com/leagues/87/overview/laliga,LaLiga - Spain,https://images.fotmob.com/image_resources/logo...


In [4]:
def extract_teams_5top_fotmob(season_in, season_out):
    df_top5= extract_5top_league_fotmob()

    df_all_teams_league = []

    for league_id in tqdm(df_top5["league_id"].unique()):
        
        url = f"https://www.fotmob.com/api/data/leagues?id={league_id}&ccode3=ESP&season={season_in}%2F{season_out}"
        response = requests.get(url)
        
        if response.status_code != 200:
            continue
        
        data = response.json()

        try:
            table = data['overview']['table'][0]['data']['table']['all']
        except:
            continue

        df_liga_teams = pd.DataFrame(table)[['name', 'shortName', 'id', 'pageUrl']]

        df_liga_teams = df_liga_teams.rename(columns={'name': 'name_team_fotmob','shortName': 'shortname_team_fotmob','id': 'id_team_fotmob','pageUrl': 'url_team_fotmob' })

        df_liga_teams['url_team_fotmob'] = 'https://www.fotmob.com' + df_liga_teams['url_team_fotmob']
        

    
        df_liga_teams['logo_team_fotmob'] = df_liga_teams['id_team_fotmob'].apply(lambda x: f"https://images.fotmob.com/image_resources/logo/teamlogo/{x}.png")

        df_liga_teams['season'] = data['details']['selectedSeason']
        df_liga_teams['id_league_fotmob'] = data['overview']['table'][0]['data']['leagueId']
        df_liga_teams['name_league_fotmob'] = data['overview']['table'][0]['data']['leagueName']
        df_liga_teams['country_code_fotmob'] = data['overview']['table'][0]['data']['ccode']

        df_all_teams_league.append(df_liga_teams)


    return pd.concat(df_all_teams_league, ignore_index=True)

In [5]:
df_all_teams_league= extract_teams_5top_fotmob('2025', '2026')
df_all_teams_league

100%|██████████| 5/5 [00:00<00:00,  6.35it/s]


,name_team_fotmob,shortname_team_fotmob,id_team_fotmob,url_team_fotmob,logo_team_fotmob,season,id_league_fotmob,name_league_fotmob,country_code_fotmob
0,Arsenal,Arsenal,9825,https://www.fotmob.com/teams/9825/overview/ars...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG
1,Manchester City,Man City,8456,https://www.fotmob.com/teams/8456/overview/man...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG
2,Manchester United,Man United,10260,https://www.fotmob.com/teams/10260/overview/ma...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG
3,Aston Villa,Aston Villa,10252,https://www.fotmob.com/teams/10252/overview/as...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG
4,Liverpool,Liverpool,8650,https://www.fotmob.com/teams/8650/overview/liv...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG
...,...,...,...,...,...,...,...,...,...
91,Levante,Levante,8581,https://www.fotmob.com/teams/8581/overview/lev...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP
92,Osasuna,Osasuna,8371,https://www.fotmob.com/teams/8371/overview/osa...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP
93,Mallorca,Mallorca,8661,https://www.fotmob.com/teams/8661/overview/mal...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP
94,Girona,Girona,7732,https://www.fotmob.com/teams/7732/overview/girona,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP


In [6]:
def extract_venues_info_teams_fotmob(season_in, season_out):
    df_all_teams_league= extract_teams_5top_fotmob(season_in, season_out)
    rows = []

    for team_id in tqdm(df_all_teams_league['id_team_fotmob'].astype(str)):
        try:
            url = f"https://www.fotmob.com/api/data/teams?id={team_id}&ccode3=ESP&season={season_in}%{season_out}"
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            data_team = response.json()

            venue = data_team.get('overview', {}).get('venue', None)

            if not venue:
                continue

            widget = venue.get('widget', {})
            stats = dict(venue.get('statPairs', []))

            row = {
                'id_team_fotmob': team_id,
                'name_venue': widget.get('name'),
                'city_venue': widget.get('city'),
                'lat_venue': widget.get('location', [None, None])[0],
                'lon_venue': widget.get('location', [None, None])[1],
                **stats
            }

            rows.append(row)

        except Exception as e:
            print(f"Error con team_id {team_id}: {e}")
            continue

    return pd.DataFrame(rows)

In [7]:
def prepare_info_all_teams_fotmob(df_all_teams_league, season_in, season_out , rute_path_stadia):
    df_all_teams_league["id_team_fotmob"] = df_all_teams_league["id_team_fotmob"].astype(str)

    df_venues= extract_venues_info_teams_fotmob(season_in, season_out)
    df_venues["id_team_fotmob"] = df_venues["id_team_fotmob"].astype(str)

    df_all_teams_league_venues = df_all_teams_league.merge(df_venues, on='id_team_fotmob', how='left')

    df_stadia = pd.read_excel(rute_path_stadia)
    df_all_teams_league_venues_photo = df_all_teams_league_venues.merge(df_stadia, on='name_team_fotmob', how='left')
    return df_all_teams_league_venues_photo


In [8]:
df_all_teams_league_venues_photo= prepare_info_all_teams_fotmob(df_all_teams_league, '2025', '2026', "links_photos_stadia_5league.xlsx" )
df_all_teams_league_venues_photo

100%|██████████| 96/96 [00:33<00:00,  2.83it/s]


,name_team_fotmob,shortname_team_fotmob,id_team_fotmob,url_team_fotmob,logo_team_fotmob,season,id_league_fotmob,name_league_fotmob,country_code_fotmob,name_venue,city_venue,lat_venue,lon_venue,Surface,Capacity,Opened,url_photo_stadium
0,Arsenal,Arsenal,9825,https://www.fotmob.com/teams/9825/overview/ars...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Emirates Stadium,London,51.555074614,-0.108457804,Grass,60704,2006,https://cdn.resfu.com/img_data/estadios/origin...
1,Manchester City,Man City,8456,https://www.fotmob.com/teams/8456/overview/man...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Etihad Stadium,Manchester,53.483152259,-2.200409174,Grass,61470,2002,https://cdn.resfu.com/img_data/estadios/origin...
2,Manchester United,Man United,10260,https://www.fotmob.com/teams/10260/overview/ma...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Old Trafford,Manchester,53.463094458,-2.291373610,Grass,74244,1910,https://cdn.resfu.com/img_data/estadios/origin...
3,Aston Villa,Aston Villa,10252,https://www.fotmob.com/teams/10252/overview/as...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Villa Park,Birmingham,52.509090736,-1.884841919,Grass,43205,1897,https://cdn.resfu.com/img_data/estadios/origin...
4,Liverpool,Liverpool,8650,https://www.fotmob.com/teams/8650/overview/liv...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Anfield,Liverpool,53.430827885,-2.960852981,Grass,61276,1884,https://cdn.resfu.com/img_data/estadios/origin...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,Levante,Levante,8581,https://www.fotmob.com/teams/8581/overview/lev...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Estadio Ciudad de Valencia,Valencia,39.494685764,-0.363820195,Grass,25534,1969,https://cdn.resfu.com/img_data/estadios/origin...
92,Osasuna,Osasuna,8371,https://www.fotmob.com/teams/8371/overview/osa...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Estadio El Sadar,Pamplona (Iruñea),42.796676994,-1.637141258,Grass,23576,1967,https://cdn.resfu.com/img_data/estadios/origin...
93,Mallorca,Mallorca,8661,https://www.fotmob.com/teams/8661/overview/mal...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Estadi Mallorca Son Moix,Palma de Mallorca,39.589939598,2.630077600,Grass,26020,1999,https://cdn.resfu.com/img_data/estadios/origin...
94,Girona,Girona,7732,https://www.fotmob.com/teams/7732/overview/girona,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Estadi Municipal de Montilivi,Girona,41.961195406,2.828495353,Grass,14500,1970,https://cdn.resfu.com/img_data/estadios/origin...


INFO TEAMS BY LEAGUE

In [9]:
laliga=df_all_teams_league_venues_photo[df_all_teams_league_venues_photo['name_league_fotmob']=='LaLiga'].reset_index(drop=True)
laliga

,name_team_fotmob,shortname_team_fotmob,id_team_fotmob,url_team_fotmob,logo_team_fotmob,season,id_league_fotmob,name_league_fotmob,country_code_fotmob,name_venue,city_venue,lat_venue,lon_venue,Surface,Capacity,Opened,url_photo_stadium
0,Barcelona,Barcelona,8634,https://www.fotmob.com/teams/8634/overview/bar...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Spotify Camp Nou,Barcelona,41.380890137,2.122812867,Grass,99787,1957,https://cdn.resfu.com/img_data/estadios/origin...
1,Real Madrid,Real Madrid,8633,https://www.fotmob.com/teams/8633/overview/rea...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Estadio Bernabéu,Madrid,40.453068279,-3.688353896,Grass,83186,1947,https://cdn.resfu.com/img_data/estadios/origin...
2,Villarreal,Villarreal,10205,https://www.fotmob.com/teams/10205/overview/vi...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Estadio de la Cerámica,Villarreal,39.944119181,-0.103474259,Grass,24500,1923,https://cdn.resfu.com/img_data/estadios/origin...
3,Atletico Madrid,Atletico Madrid,9906,https://www.fotmob.com/teams/9906/overview/atl...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Riyadh Air Metropolitano,Madrid,40.436225928,-3.599460125,Grass,70460,1994,https://cdn.resfu.com/img_data/estadios/origin...
4,Real Betis,Real Betis,8603,https://www.fotmob.com/teams/8603/overview/rea...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Estadio La Cartuja de Sevilla,Sevilla,37.417208903,-6.004661322,Grass,72000,1999,https://cdn.resfu.com/img_data/estadios/origin...
5,Celta Vigo,Celta Vigo,9910,https://www.fotmob.com/teams/9910/overview/cel...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Estadio Abanca-Balaídos,Vigo,42.211851813,-8.739720583,Grass,24870,1928,https://cdn.resfu.com/img_data/estadios/origin...
6,Getafe,Getafe,8305,https://www.fotmob.com/teams/8305/overview/getafe,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Estadio Coliseum,Getafe,40.325728646,-3.714964092,Grass,17393,1998,https://cdn.resfu.com/img_data/estadios/origin...
7,Rayo Vallecano,Rayo Vallecano,8370,https://www.fotmob.com/teams/8370/overview/ray...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Estadio de Vallecas,Madrid,40.391822825,-3.658589423,Grass,15500,1976,https://cdn.resfu.com/img_data/estadios/origin...
8,Valencia,Valencia,10267,https://www.fotmob.com/teams/10267/overview/va...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Estadio de Mestalla,Valencia,39.474601503,-0.358257294,Grass,49430,1923,https://cdn.resfu.com/img_data/estadios/origin...
9,Real Sociedad,Real Sociedad,8560,https://www.fotmob.com/teams/8560/overview/rea...,https://images.fotmob.com/image_resources/logo...,2025/2026,87,LaLiga,ESP,Estadio Municipal de Anoeta,Donostia-San Sebastián,43.301375816,-1.973601580,Grass,40000,1993,https://cdn.resfu.com/img_data/estadios/origin...


ALL FIXTURES LEAGUES FOTMOB

In [10]:
def extract_fixtures_5top_fotmob(season_in, season_out):
    df_top5= extract_5top_league_fotmob()
    df_all_fixtures = []

    for league_id in tqdm(df_top5["league_id"].unique()):
        
        url =f"https://www.fotmob.com/api/data/leagues?id={league_id}&ccode3=ESP&season={season_in}%2F{season_out}"
        
        response = requests.get(url)
        
        if response.status_code != 200:
            continue
        
        data = response.json()

        try:
            fixtures = data["fixtures"]["allMatches"]
        except KeyError:
            continue

        # normalizar
        df_fixtures = pd.json_normalize(fixtures)

        # añadir info de liga
        try:
            df_fixtures["id_league_fotmob"] = league_id
            df_fixtures["name_league_fotmob"] = data["overview"]["table"][0]["data"]["leagueName"]
            df_fixtures["season"] = data['details']['selectedSeason']
            df_fixtures["country_code_fotmob"] = data["overview"]["table"][0]["data"]["ccode"]
        except:
            pass

        df_all_fixtures.append(df_fixtures)
    df_all_fixtures= pd.concat(df_all_fixtures, ignore_index=True)

    all_fixtures_fotmob = df_all_fixtures.drop(columns=['roundName', 'home.shortName', 'away.shortName', 'status.timezone',
                                                    'status.reason.shortKey'  ],errors= 'ignore')
    all_fixtures_fotmob['pageUrl'] = "https://www.fotmob.com" +  all_fixtures_fotmob['pageUrl']
    all_fixtures_fotmob['logo_team_home'] = all_fixtures_fotmob['home.id'].apply(lambda x: f"https://images.fotmob.com/image_resources/logo/teamlogo/{x}.png")
    all_fixtures_fotmob['logo_team_away'] = all_fixtures_fotmob['away.id'].apply(lambda x: f"https://images.fotmob.com/image_resources/logo/teamlogo/{x}.png")
    
    # Convertir a datetime con zona UTC
    all_fixtures_fotmob["status.utcTime"] = pd.to_datetime(all_fixtures_fotmob["status.utcTime"], utc=True)

    fecha = pd.to_datetime(all_fixtures_fotmob["status.utcTime"], utc=True).dt.tz_convert("Europe/Madrid")

    all_fixtures_fotmob["match_date"] = fecha.dt.date
    all_fixtures_fotmob["match_time"] = fecha.dt.strftime("%H:%M")

    all_fixtures_fotmob = all_fixtures_fotmob.drop(columns=["status.utcTime"])

    return  all_fixtures_fotmob

In [11]:
df_all_fixtures=extract_fixtures_5top_fotmob('2025', '2026') 
df_all_fixtures

100%|██████████| 5/5 [00:01<00:00,  2.70it/s]


,round,pageUrl,id,home.name,home.id,away.name,away.id,status.finished,status.started,status.cancelled,...,status.reason.long,status.reason.longKey,id_league_fotmob,name_league_fotmob,season,country_code_fotmob,logo_team_home,logo_team_away,match_date,match_time
0,1,https://www.fotmob.com/matches/liverpool-vs-af...,4813374,Liverpool,8650,AFC Bournemouth,8678,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2025-08-15,21:00
1,1,https://www.fotmob.com/matches/aston-villa-vs-...,4813375,Aston Villa,10252,Newcastle United,10261,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2025-08-16,13:30
2,1,https://www.fotmob.com/matches/fulham-vs-brigh...,4813376,Brighton & Hove Albion,10204,Fulham,9879,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2025-08-16,16:00
3,1,https://www.fotmob.com/matches/sunderland-vs-w...,4813378,Sunderland,8472,West Ham United,8654,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2025-08-16,16:00
4,1,https://www.fotmob.com/matches/burnley-vs-tott...,4813379,Tottenham Hotspur,8586,Burnley,8191,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2025-08-16,16:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1747,38,https://www.fotmob.com/matches/mallorca-vs-rea...,4837485,Mallorca,8661,Real Oviedo,8670,True,True,False,...,Full-Time,finished,87,LaLiga,2025/2026,ESP,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2026-05-23,21:00
1748,38,https://www.fotmob.com/matches/levante-vs-real...,4837481,Real Betis,8603,Levante,8581,True,True,False,...,Full-Time,finished,87,LaLiga,2025/2026,ESP,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2026-05-23,21:00
1749,38,https://www.fotmob.com/matches/athletic-club-v...,4837486,Real Madrid,8633,Athletic Club,8315,True,True,False,...,Full-Time,finished,87,LaLiga,2025/2026,ESP,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2026-05-23,21:00
1750,38,https://www.fotmob.com/matches/barcelona-vs-va...,4837488,Valencia,10267,Barcelona,8634,True,True,False,...,Full-Time,finished,87,LaLiga,2025/2026,ESP,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2026-05-23,21:00


In [12]:
df_all_fixtures[df_all_fixtures['name_league_fotmob']=='Premier League']

,round,pageUrl,id,home.name,home.id,away.name,away.id,status.finished,status.started,status.cancelled,...,status.reason.long,status.reason.longKey,id_league_fotmob,name_league_fotmob,season,country_code_fotmob,logo_team_home,logo_team_away,match_date,match_time
0,1,https://www.fotmob.com/matches/liverpool-vs-af...,4813374,Liverpool,8650,AFC Bournemouth,8678,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2025-08-15,21:00
1,1,https://www.fotmob.com/matches/aston-villa-vs-...,4813375,Aston Villa,10252,Newcastle United,10261,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2025-08-16,13:30
2,1,https://www.fotmob.com/matches/fulham-vs-brigh...,4813376,Brighton & Hove Albion,10204,Fulham,9879,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2025-08-16,16:00
3,1,https://www.fotmob.com/matches/sunderland-vs-w...,4813378,Sunderland,8472,West Ham United,8654,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2025-08-16,16:00
4,1,https://www.fotmob.com/matches/burnley-vs-tott...,4813379,Tottenham Hotspur,8586,Burnley,8191,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2025-08-16,16:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,38,https://www.fotmob.com/matches/manchester-city...,4813750,Manchester City,8456,Aston Villa,10252,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2026-05-24,17:00
376,38,https://www.fotmob.com/matches/afc-bournemouth...,4813751,Nottingham Forest,10203,AFC Bournemouth,8678,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2026-05-24,17:00
377,38,https://www.fotmob.com/matches/chelsea-vs-sund...,4813752,Sunderland,8472,Chelsea,8455,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2026-05-24,17:00
378,38,https://www.fotmob.com/matches/tottenham-hotsp...,4813753,Tottenham Hotspur,8586,Everton,8668,True,True,False,...,Full-Time,finished,47,Premier League,2025/2026,ENG,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/logo...,2026-05-24,17:00


WHOSCORED

In [28]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException, NoSuchElementException


# =========================================================
# EXTRAER METADATOS DESDE LA URL
# /regions/206/tournaments/4/seasons/10803/stages/24622/fixtures/spain-laliga-2025-2026
# =========================================================
def parse_url_metadata(url):
    patterns = {
        "regionId":     r"/regions/(\d+)",
        "tournamentId": r"/tournaments/(\d+)",
        "seasonId":     r"/seasons/(\d+)",
        "stageId":      r"/stages/(\d+)",
    }
    meta = {
        k: (re.search(p, url).group(1) if re.search(p, url) else None)
        for k, p in patterns.items()
    }

    # Slug final: "spain-laliga-2025-2026"
    slug_match = re.search(r"/fixtures/([^/?#]+)", url)
    if slug_match:
        parts = slug_match.group(1).split("-")
        # Los últimos dos tokens son el año (ej. 2025, 2026)
        meta["season"]     = f"{parts[-2]}-{parts[-1]}"
        meta["leagueName"] = " ".join(p.capitalize() for p in parts[:-2])
    else:
        meta["season"]     = None
        meta["leagueName"] = None

    return meta


# =========================================================
# OBTENER MES ACTUAL DEL CALENDARIO
# =========================================================
def get_month(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    el = soup.select_one("#toggleCalendar span")
    return el.text.strip() if el else "unknown"


# =========================================================
# FIRMA DEL DOM: mes + primer match ID + total de partidos
# =========================================================
def get_dom_signature(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")

    month_el = soup.select_one("#toggleCalendar span")
    month = month_el.text.strip() if month_el else "unknown"

    first_score = soup.select_one("a[id^='scoresBtn-']")
    first_id = first_score["id"] if first_score else "none"

    total = len(soup.select("a[id^='scoresBtn-']"))

    return f"{month}|{first_id}|{total}"


# =========================================================
# CLICK EN BOTÓN Y ESPERAR CAMBIO REAL DE DOM
# =========================================================
def click_and_wait(driver, btn_id, timeout=15):
    old_sig = get_dom_signature(driver)

    try:
        btn = driver.find_element(By.ID, btn_id)
        driver.execute_script("arguments[0].click();", btn)
    except NoSuchElementException:
        return False

    try:
        WebDriverWait(driver, timeout).until(
            lambda d: get_dom_signature(d) != old_sig
        )
        time.sleep(0.5)
        return True
    except TimeoutException:
        return False


# =========================================================
# EXTRAER PARTIDOS DEL DOM RENDERIZADO
# =========================================================
def extract_matches_from_dom(driver, calendar_month, url_meta):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    rows = []

    for day_block in soup.select("div.Accordion-module_accordion__UuHD0"):
        date_el = day_block.select_one("div.Accordion-module_header__HqzWD span")
        day_str = date_el.text.strip() if date_el else "unknown"

        for match in day_block.select("div.Match-module_match__XlKTY"):

            score_link = match.select_one("a[id^='scoresBtn-']")
            if not score_link:
                continue
            match_id  = score_link["id"].replace("scoresBtn-", "")
            match_url = "https://www.whoscored.com" + score_link.get("href", "")

            status_el = match.select_one("span[class*='Match-module_FT']")
            status    = status_el.text.strip() if status_el else ""

            score_spans = score_link.select("span")
            home_score  = score_spans[0].text.strip() if len(score_spans) > 0 else None
            away_score  = score_spans[1].text.strip() if len(score_spans) > 1 else None

            team_links = match.select("a.Match-module_teamNameText__Dqv-G")
            home_team  = team_links[0].text.strip() if len(team_links) > 0 else None
            away_team  = team_links[1].text.strip() if len(team_links) > 1 else None

            home_team_id = away_team_id = None
            if len(team_links) > 0:
                m = re.search(r"/teams/(\d+)/", team_links[0].get("href", ""))
                home_team_id = m.group(1) if m else None
            if len(team_links) > 1:
                m = re.search(r"/teams/(\d+)/", team_links[1].get("href", ""))
                away_team_id = m.group(1) if m else None

            rows.append({
                # metadatos del torneo
                "regionId":     url_meta["regionId"],
                "tournamentId": url_meta["tournamentId"],
                "seasonId":     url_meta["seasonId"],
                "stageId":      url_meta["stageId"],
                "leagueName":   url_meta["leagueName"],
                "season":       url_meta["season"],
                # partido
                "match_id":       match_id,
                "match_url":      match_url,
                "calendar_month": calendar_month,
                "day_str":        day_str,
                "status":         status,
                "homeTeamName":   home_team,
                "awayTeamName":   away_team,
                "homeTeamId":     home_team_id,
                "awayTeamId":     away_team_id,
                "homeScore":      home_score,
                "awayScore":      away_score,
            })

    df = pd.DataFrame(rows)

    if not df.empty:
        df["match_date"] = pd.to_datetime(
            df["day_str"], format="%A, %b %d %Y", errors="coerce"
        ).dt.date
        df = df.drop(columns=["day_str"])

        for col in ["homeScore", "awayScore"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        # Logos — solo cuando el ID no es None
        base = "https://d2zywfiolv4f83.cloudfront.net/img/teams/"
        df["homeLogo"] = df["homeTeamId"].apply(
            lambda x: f"{base}{x}.png" if pd.notna(x) else None
        )
        df["awayLogo"] = df["awayTeamId"].apply(
            lambda x: f"{base}{x}.png" if pd.notna(x) else None
        )

    return df


# =========================================================
# IR AL PRIMER MES
# =========================================================
def go_to_first_month(driver):
    print("⏪ Navegando al primer mes...")
    steps = 0
    max_steps = 24

    while steps < max_steps:
        changed = click_and_wait(driver, "dayChangeBtn-prev", timeout=12)
        if not changed:
            print(f"✅ Primer mes alcanzado tras {steps} clicks prev")
            return
        steps += 1
        print(f"  ← {get_month(driver)}")

    print("⚠️  Límite de pasos alcanzado")


# =========================================================
# SCRAPING HACIA ADELANTE MES A MES
# =========================================================
def scrape_forward(driver, url_meta):
    all_dfs   = []
    seen_sigs = set()

    while True:
        month = get_month(driver)
        sig   = get_dom_signature(driver)

        print(f"\n📅 Extrayendo: {month}")

        if sig in seen_sigs:
            print("🏁 Contenido DOM repetido — fin del calendario")
            break
        seen_sigs.add(sig)

        df = extract_matches_from_dom(driver, month, url_meta)

        if not df.empty:
            all_dfs.append(df)
            print(f"  ✓ {len(df)} partidos")
        else:
            print("  ✗ Sin partidos en el DOM")

        changed = click_and_wait(driver, "dayChangeBtn-next", timeout=12)
        if not changed:
            print("🏁 DOM no cambió tras click next — fin del calendario")
            break

    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()


# =========================================================
# MAIN
# =========================================================
def scrape_whoscored(url, headless=True):
    url_meta = parse_url_metadata(url)
    print(f"📋 Liga: {url_meta['leagueName']} | Temporada: {url_meta['season']} "
          f"| tournamentId: {url_meta['tournamentId']} | seasonId: {url_meta['seasonId']}")

    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)

    try:
        print(f"🌐 Cargando: {url}")
        driver.get(url)

        WebDriverWait(driver, 30).until(
            lambda d: len(BeautifulSoup(d.page_source, "html.parser")
                         .select("a[id^='scoresBtn-']")) > 0
        )
        print("✅ Página cargada y partidos visibles en DOM")

        go_to_first_month(driver)
        df = scrape_forward(driver, url_meta)

    finally:
        driver.quit()

    print(f"\n🎉 Total partidos: {len(df)}")
    return df

MATCHES 2025-2026 LA LIGA

In [30]:
url = "https://www.whoscored.com/regions/206/tournaments/4/seasons/10803/stages/24622/fixtures/spain-laliga-2025-2026"
df_laliga = scrape_whoscored(url, headless=False)
df_laliga

📋 Liga: Spain Laliga | Temporada: 2025-2026 | tournamentId: 4 | seasonId: 10803
🌐 Cargando: https://www.whoscored.com/regions/206/tournaments/4/seasons/10803/stages/24622/fixtures/spain-laliga-2025-2026
✅ Página cargada y partidos visibles en DOM
⏪ Navegando al primer mes...
  ← Apr 2026
  ← Mar 2026
  ← Feb 2026
  ← Jan 2026
  ← Dec 2025
  ← Nov 2025
  ← Oct 2025
  ← Sep 2025
  ← Aug 2025
✅ Primer mes alcanzado tras 9 clicks prev

📅 Extrayendo: Aug 2025
  ✓ 31 partidos

📅 Extrayendo: Sep 2025
  ✓ 39 partidos

📅 Extrayendo: Oct 2025
  ✓ 31 partidos

📅 Extrayendo: Nov 2025
  ✓ 38 partidos

📅 Extrayendo: Dec 2025
  ✓ 32 partidos

📅 Extrayendo: Jan 2026
  ✓ 43 partidos

📅 Extrayendo: Feb 2026
  ✓ 40 partidos

📅 Extrayendo: Mar 2026
  ✓ 36 partidos

📅 Extrayendo: Apr 2026
  ✓ 40 partidos

📅 Extrayendo: May 2026
  ✓ 50 partidos
🏁 DOM no cambió tras click next — fin del calendario

🎉 Total partidos: 380


,regionId,tournamentId,seasonId,stageId,leagueName,season,match_id,match_url,calendar_month,status,homeTeamName,awayTeamName,homeTeamId,awayTeamId,homeScore,awayScore,match_date,homeLogo,awayLogo
0,206,4,10803,24622,Spain Laliga,2025-2026,1913916,https://www.whoscored.com/matches/1913916/live...,Aug 2025,FT,Girona,Rayo Vallecano,2783,64,1,3,2025-08-15,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
1,206,4,10803,24622,Spain Laliga,2025-2026,1913892,https://www.whoscored.com/matches/1913892/live...,Aug 2025,FT,Villarreal,Real Oviedo,839,61,2,0,2025-08-15,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
2,206,4,10803,24622,Spain Laliga,2025-2026,1913918,https://www.whoscored.com/matches/1913918/live...,Aug 2025,FT,Mallorca,Barcelona,51,65,0,3,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
3,206,4,10803,24622,Spain Laliga,2025-2026,1913913,https://www.whoscored.com/matches/1913913/live...,Aug 2025,FT,Deportivo Alaves,Levante,60,832,2,1,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
4,206,4,10803,24622,Spain Laliga,2025-2026,1913889,https://www.whoscored.com/matches/1913889/live...,Aug 2025,FT,Valencia,Real Sociedad,55,68,1,1,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,206,4,10803,24622,Spain Laliga,2025-2026,1914258,https://www.whoscored.com/matches/1914258/live...,May 2026,FT,Valencia,Barcelona,55,65,3,1,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
376,206,4,10803,24622,Spain Laliga,2025-2026,1914259,https://www.whoscored.com/matches/1914259/live...,May 2026,FT,Girona,Elche,2783,833,1,1,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
377,206,4,10803,24622,Spain Laliga,2025-2026,1914260,https://www.whoscored.com/matches/1914260/live...,May 2026,FT,Getafe,Osasuna,819,131,1,0,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
378,206,4,10803,24622,Spain Laliga,2025-2026,1914261,https://www.whoscored.com/matches/1914261/live...,May 2026,FT,Mallorca,Real Oviedo,51,61,3,0,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...


In [34]:
df_laliga.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   regionId        380 non-null    object        
 1   tournamentId    380 non-null    object        
 2   seasonId        380 non-null    object        
 3   stageId         380 non-null    object        
 4   leagueName      380 non-null    object        
 5   season          380 non-null    object        
 6   match_id        380 non-null    object        
 7   match_url       380 non-null    object        
 8   calendar_month  380 non-null    object        
 9   status          380 non-null    object        
 10  homeTeamName    380 non-null    object        
 11  awayTeamName    380 non-null    object        
 12  homeTeamId      380 non-null    object        
 13  awayTeamId      380 non-null    object        
 14  homeScore       380 non-null    int64         
 15  awaySc

In [198]:
LEAGUE_CALENDARS = {
    "Spain Laliga": {
        "matchweeks": [
            ("2025-08-15", "2025-08-19", 1),
            ("2025-08-22", "2025-08-25", 2),
            ("2025-08-29", "2025-09-01", 3),
            ("2025-09-12", "2025-09-15", 4),
            ("2025-09-19", "2025-09-22", 5),
            ("2025-09-23", "2025-09-25", 6),
            ("2025-09-26", "2025-09-30", 7),
            ("2025-10-03", "2025-10-06", 8),
            ("2025-10-17", "2025-10-20", 9),
            ("2025-10-24", "2025-10-27", 10),
            ("2025-10-31", "2025-11-03", 11),
            ("2025-11-07", "2025-11-10", 12),
            ("2025-11-21", "2025-11-24", 13),
            ("2025-11-28", "2025-12-01", 14),
            ("2025-12-05", "2025-12-08", 15),
            ("2025-12-12", "2025-12-15", 16),
            ("2025-12-19", "2025-12-22", 17),
            ("2026-01-02", "2026-01-05", 18),
            ("2026-01-09", "2026-01-12", 19),
            ("2026-01-16", "2026-01-19", 20),
            ("2026-01-23", "2026-01-26", 21),
            ("2026-01-30", "2026-02-02", 22),
            ("2026-02-06", "2026-02-09", 23),
            ("2026-02-13", "2026-02-16", 24),
            ("2026-02-20", "2026-02-23", 25),
            ("2026-02-27", "2026-03-02", 26),
            ("2026-03-06", "2026-03-09", 27),
            ("2026-03-13", "2026-03-16", 28),
            ("2026-03-20", "2026-03-23", 29),
            ("2026-04-03", "2026-04-06", 30),
            ("2026-04-10", "2026-04-13", 31),
            ("2026-04-21", "2026-04-23", 33),
            ("2026-04-24", "2026-04-27", 32),
            ("2026-05-01", "2026-05-04", 34),
            ("2026-05-08", "2026-05-11", 35),
            ("2026-05-12", "2026-05-14", 36),
            ("2026-05-15", "2026-05-18", 37),
            ("2026-05-22", "2026-05-24", 38),
        ],
        "exceptions": {
            "1913937": 6,
            "1914066": 19,
            "1914062": 19,
            "1914035": 16,
            "1914092": 23,
        },
    },

    "England Premier League": {
        "matchweeks": [          
            ("2025-08-15", "2025-08-18", 1),
            ("2025-08-22", "2025-08-25", 2),
            ("2025-08-29", "2025-09-01", 3),
            ("2025-09-13", "2025-09-15", 4),
            ("2025-09-20", "2025-09-22", 5),
            ("2025-09-27", "2025-09-29", 6),
            ("2025-10-03", "2025-10-06", 7),
            ("2025-10-18", "2025-10-20", 8),
            ("2025-10-24", "2025-10-27", 9),
            ("2025-11-01", "2025-11-03", 10),
            ("2025-11-08", "2025-11-10", 11),
            ("2025-11-22", "2025-11-24", 12),
            ("2025-11-29", "2025-12-01", 13),
            ("2025-12-02", "2025-12-05", 14),   # Intersemanal
            ("2025-12-06", "2025-12-08", 15),
            ("2025-12-13", "2025-12-15", 16),
            ("2025-12-20", "2025-12-22", 17),
            ("2025-12-26", "2025-12-29", 18),
            ("2025-12-30", "2026-01-01", 19),   # Intersemanal
            ("2026-01-03", "2026-01-05", 20),
            ("2026-01-06", "2026-01-08", 21),
            ("2026-01-17", "2026-01-19", 22),
            ("2026-01-24", "2026-01-26", 23),
            ("2026-01-31", "2026-02-02", 24),
            ("2026-02-06", "2026-02-08", 25),   # Intersemanal
            ("2026-02-10", "2026-02-12", 26),
            ("2026-02-21", "2026-02-23", 27),
            ("2026-02-27", "2026-03-01", 28),   # Intersemanal
            ("2026-03-03", "2026-03-05", 29),
            ("2026-03-14", "2026-03-16", 30),
            ("2026-03-20", "2026-03-23", 31),
            ("2026-04-10", "2026-04-13", 32),
            ("2026-04-18", "2026-04-20", 33),
            ("2026-04-21", "2026-04-27", 34),
            ("2026-05-01", "2026-05-04", 35),
            ("2026-05-09", "2026-05-11", 36),
            ("2026-05-15", "2026-05-19", 37),   # Intersemanal
            ("2026-05-24", "2026-05-25", 38),

        ],
        "exceptions": {
            "1903466": 31,
            "1903469": 31
           
        },
    },

    "Germany Bundesliga": {
        "matchweeks": [          
            ("2025-08-22", "2025-08-25", 1),
            ("2025-08-29", "2025-09-01", 2),
            ("2025-09-12", "2025-09-15", 3),
            ("2025-09-19", "2025-09-22", 4),
            ("2025-09-26", "2025-09-29", 5),
            ("2025-10-03", "2025-10-06", 6),
            ("2025-10-17", "2025-10-20", 7),
            ("2025-10-24", "2025-10-27", 8),
            ("2025-10-31", "2025-11-03", 9),
            ("2025-11-07", "2025-11-10", 10),
            ("2025-11-21", "2025-11-24", 11),
            ("2025-11-28", "2025-12-01", 12),
            ("2025-12-05", "2025-12-08", 13),
            ("2025-12-12", "2025-12-15", 14),
            ("2025-12-19", "2025-12-22", 15),
            ("2026-01-09", "2026-01-12", 16),
            ("2026-01-13", "2026-01-15", 17),  
            ("2026-01-16", "2026-01-19", 18),
            ("2026-01-23", "2026-01-26", 19),  
            ("2026-01-30", "2026-02-02", 20),
            ("2026-02-06", "2026-02-09", 21),
            ("2026-02-13", "2026-02-16", 22),
            ("2026-02-20", "2026-02-23", 23),
            ("2026-02-27", "2026-03-02", 24),
            ("2026-03-06", "2026-03-09", 25),
            ("2026-03-13", "2026-03-16", 26),
            ("2026-04-20", "2026-04-23", 27),
            ("2026-04-04", "2026-04-06", 28),
            ("2026-04-10", "2026-04-13", 29),
            ("2026-04-17", "2026-04-20", 30),
            ("2026-04-24", "2026-04-27", 31),
            ("2026-05-02", "2026-05-04", 32),
            ("2026-05-08", "2026-05-11", 33),
            ("2026-05-16", "2026-05-17", 34),

        ],
        "exceptions": {
           "1910684": 16,
           "1910688": 16,
           "1910799": 17,
           
           
        },
    },

    "Italy Serie A": {
        "matchweeks": [          
            ("2025-08-23", "2025-08-25", 1),
            ("2025-08-29", "2025-09-01", 2),
            ("2025-09-13", "2025-09-15", 3),
            ("2025-09-19", "2025-09-22", 4),
            ("2025-09-27", "2025-09-29", 5),
            ("2025-10-03", "2025-10-06", 6),
            ("2025-10-18", "2025-10-20", 7),
            ("2025-10-24", "2025-10-27", 8),
            ("2025-10-28", "2025-10-30", 9),
            ("2025-11-01", "2025-11-03", 10),
            ("2025-11-07", "2025-11-10", 11),
            ("2025-11-21", "2025-11-24", 12),
            ("2025-11-28", "2025-12-01", 13),
            ("2025-12-06", "2025-12-09", 14),
            ("2025-12-12", "2025-12-15", 15),
            ("2025-12-20", "2025-12-22", 16),
            ("2026-12-27", "2026-12-29", 17),
            ("2026-01-02", "2026-01-04", 18),
            ("2026-01-06", "2026-01-08", 19),
            ("2026-01-10", "2026-01-12", 20),
            ("2026-01-16", "2026-01-19", 21),
            ("2026-01-23", "2026-02-26", 22),
            ("2026-01-30", "2026-02-03", 23),
            ("2026-02-06", "2026-02-09", 24),
            ("2026-02-13", "2026-02-16", 25),
            ("2026-02-20", "2026-02-23", 26),
            ("2026-02-27", "2026-03-02", 27),
            ("2026-03-06", "2026-03-09", 28),
            ("2026-03-13", "2026-03-16", 29),
            ("2026-03-20", "2026-03-23", 30),
            ("2026-04-04", "2026-04-06", 31),
            ("2026-04-10", "2026-04-13", 32),
            ("2026-04-17", "2026-04-20", 33),
            ("2026-04-24", "2026-04-27", 34),
            ("2026-05-01", "2026-05-04", 35),
            ("2026-05-08", "2026-05-11", 36),
            ("2026-05-17", "2026-05-18", 37),  # intersemanal posible
            ("2026-05-22", "2026-05-25", 38),

        ],
        "exceptions": {
            "1901309": 16,
            "1901306": 16,
            "1901311": 16,
            "1901303": 16,
            "1901358": 24
 
        },
    },

    "France Ligue 1": {
        "matchweeks": [          
            ("2025-08-15", "2025-08-17", 1),
            ("2025-08-22", "2025-08-24", 2),
            ("2025-08-29", "2025-08-31", 3),
            ("2025-09-12", "2025-09-14", 4),
            ("2025-09-19", "2025-09-22", 5),
            ("2025-09-26", "2025-09-28", 6),
            ("2025-10-03", "2025-10-05", 7),
            ("2025-10-17", "2025-10-19", 8),
            ("2025-10-24", "2025-10-26", 9),
            ("2025-10-29", "2025-10-30", 10),
            ("2025-11-01", "2025-11-03", 11),
            ("2025-11-07", "2025-11-09", 12),
            ("2025-11-21", "2025-11-23", 13),
            ("2025-11-28", "2025-11-30", 14),
            ("2025-12-05", "2025-12-07", 15),
            ("2025-12-12", "2025-12-14", 16),
            ("2026-01-02", "2026-01-04", 17),
            ("2026-01-16", "2026-01-18", 18),
            ("2026-01-23", "2026-01-25", 19),
            ("2026-01-30", "2026-02-01", 20),
            ("2026-02-06", "2026-02-08", 21),
            ("2026-02-13", "2026-02-15", 22),
            ("2026-02-20", "2026-02-22", 23),
            ("2026-02-27", "2026-03-01", 24),
            ("2026-03-06", "2026-03-08", 25),
            ("2026-03-13", "2026-03-15", 26),
            ("2026-03-20", "2026-03-22", 27),
            ("2026-04-03", "2026-04-05", 28),
            ("2026-04-10", "2026-04-12", 29),
            ("2026-04-17", "2026-04-19", 30),
            ("2026-04-24", "2026-04-26", 31),
            ("2026-05-01", "2026-05-03", 32),
            ("2026-05-08", "2026-05-10", 33),
            ("2026-05-17", "2026-05-18", 34),
           

        ],
        "exceptions": {
            "1911504": 26,
            "1911537": 29,
            "1911538": 29

        },
    },
}

In [126]:
def transformation_fixtures_matchweek(df):
    df = df.copy()
    df["match_date"] = pd.to_datetime(df["match_date"])

    league = df["leagueName"].iat[0]

    if league not in LEAGUE_CALENDARS:
        raise ValueError(f"No hay calendario definido para '{league}'")

    matchweeks = LEAGUE_CALENDARS[league]["matchweeks"]
    exceptions = LEAGUE_CALENDARS[league]["exceptions"]

    df["matchweek_id"] = np.nan

    # Asignación por fechas
    for start, end, week in matchweeks:
        mask = df["match_date"].between(start, end)
        df.loc[mask, "matchweek_id"] = week

    # Corrección de partidos aplazados
    for match_id, week in exceptions.items():
        df.loc[df["match_id"] == match_id, "matchweek_id"] = week

    df["matchweek_id"] = df["matchweek_id"].astype("Int64")
    df["matchround"] = df["matchweek_id"].astype(str)

    return df

In [59]:
pd.set_option('display.max_columns', None)

In [127]:
df_laliga1= transformation_fixtures_matchweek(df_laliga)
df_laliga1

,regionId,tournamentId,seasonId,stageId,leagueName,season,match_id,match_url,calendar_month,status,homeTeamName,awayTeamName,homeTeamId,awayTeamId,homeScore,awayScore,match_date,homeLogo,awayLogo,matchweek_id,matchround
0,206,4,10803,24622,Spain Laliga,2025-2026,1913916,https://www.whoscored.com/matches/1913916/live...,Aug 2025,FT,Girona,Rayo Vallecano,2783,64,1,3,2025-08-15,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
1,206,4,10803,24622,Spain Laliga,2025-2026,1913892,https://www.whoscored.com/matches/1913892/live...,Aug 2025,FT,Villarreal,Real Oviedo,839,61,2,0,2025-08-15,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
2,206,4,10803,24622,Spain Laliga,2025-2026,1913918,https://www.whoscored.com/matches/1913918/live...,Aug 2025,FT,Mallorca,Barcelona,51,65,0,3,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
3,206,4,10803,24622,Spain Laliga,2025-2026,1913913,https://www.whoscored.com/matches/1913913/live...,Aug 2025,FT,Deportivo Alaves,Levante,60,832,2,1,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
4,206,4,10803,24622,Spain Laliga,2025-2026,1913889,https://www.whoscored.com/matches/1913889/live...,Aug 2025,FT,Valencia,Real Sociedad,55,68,1,1,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,206,4,10803,24622,Spain Laliga,2025-2026,1914258,https://www.whoscored.com/matches/1914258/live...,May 2026,FT,Valencia,Barcelona,55,65,3,1,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
376,206,4,10803,24622,Spain Laliga,2025-2026,1914259,https://www.whoscored.com/matches/1914259/live...,May 2026,FT,Girona,Elche,2783,833,1,1,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
377,206,4,10803,24622,Spain Laliga,2025-2026,1914260,https://www.whoscored.com/matches/1914260/live...,May 2026,FT,Getafe,Osasuna,819,131,1,0,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
378,206,4,10803,24622,Spain Laliga,2025-2026,1914261,https://www.whoscored.com/matches/1914261/live...,May 2026,FT,Mallorca,Real Oviedo,51,61,3,0,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38


In [128]:
df_laliga1[df_laliga1['matchweek_id']==38]

,regionId,tournamentId,seasonId,stageId,leagueName,season,match_id,match_url,calendar_month,status,homeTeamName,awayTeamName,homeTeamId,awayTeamId,homeScore,awayScore,match_date,homeLogo,awayLogo,matchweek_id,matchround
370,206,4,10803,24622,Spain Laliga,2025-2026,1914252,https://www.whoscored.com/matches/1914252/live...,May 2026,FT,Deportivo Alaves,Rayo Vallecano,60,64,1,2,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
371,206,4,10803,24622,Spain Laliga,2025-2026,1914253,https://www.whoscored.com/matches/1914253/live...,May 2026,FT,Real Betis,Levante,54,832,2,1,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
372,206,4,10803,24622,Spain Laliga,2025-2026,1914254,https://www.whoscored.com/matches/1914254/live...,May 2026,FT,Celta Vigo,Sevilla,62,67,1,0,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
373,206,4,10803,24622,Spain Laliga,2025-2026,1914255,https://www.whoscored.com/matches/1914255/live...,May 2026,FT,Espanyol,Real Sociedad,70,68,1,1,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
374,206,4,10803,24622,Spain Laliga,2025-2026,1914256,https://www.whoscored.com/matches/1914256/live...,May 2026,FT,Real Madrid,Athletic Club,52,53,4,2,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
375,206,4,10803,24622,Spain Laliga,2025-2026,1914258,https://www.whoscored.com/matches/1914258/live...,May 2026,FT,Valencia,Barcelona,55,65,3,1,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
376,206,4,10803,24622,Spain Laliga,2025-2026,1914259,https://www.whoscored.com/matches/1914259/live...,May 2026,FT,Girona,Elche,2783,833,1,1,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
377,206,4,10803,24622,Spain Laliga,2025-2026,1914260,https://www.whoscored.com/matches/1914260/live...,May 2026,FT,Getafe,Osasuna,819,131,1,0,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
378,206,4,10803,24622,Spain Laliga,2025-2026,1914261,https://www.whoscored.com/matches/1914261/live...,May 2026,FT,Mallorca,Real Oviedo,51,61,3,0,2026-05-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
379,206,4,10803,24622,Spain Laliga,2025-2026,1914257,https://www.whoscored.com/matches/1914257/live...,May 2026,FT,Villarreal,Atletico Madrid,839,63,5,1,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38


In [142]:
def compare_team_names(df1, col1, df2, col2, normalize=False, verbose=True):
    """
    Compara valores únicos de dos columnas de equipos.
    """

    def norm(x):
        return x.strip().lower()

    s1 = set(df1[col1].dropna().unique())
    s2 = set(df2[col2].dropna().unique())

    if normalize:
        s1 = set(norm(x) for x in s1)
        s2 = set(norm(x) for x in s2)

    only_1 = s1 - s2
    only_2 = s2 - s1
    intersection = s1 & s2

    if verbose:
        print(f"Total únicos {col1}: {len(s1)}")
        print(f"Total únicos {col2}: {len(s2)}\n")

        print(f"Solo en {col1}:")
        print(only_1 if only_1 else "None")

        print(f"\nSolo en {col2}:")
        print(only_2 if only_2 else "None")

        print(f"\nCoinciden: {len(intersection)}")

    return {
        "only_in_col1": only_1,
        "only_in_col2": only_2,
        "intersection": intersection
    }

In [147]:
MAPPING_NAMES_WHOSCORED_TO_FOTMOB= {
    #LA LIGA NONE

    #PREMIER LEAGUE
    'Wolves': 'Wolverhampton Wanderers', 
    'West Ham': 'West Ham United',
    'Brighton': 'Brighton & Hove Albion',
    'Tottenham': 'Tottenham Hotspur',
    'Newcastle': 'Newcastle United',
    'Leeds': 'Leeds United',
    'Bournemouth': 'AFC Bournemouth',

    #BUNDESLIGA
    'FC Koln': '1. FC Köln', 
    'Bayern Munich': 'Bayern München', 
    'Borussia M.Gladbach': 'Borussia Mönchengladbach',

    #SERIE A
    'AC Milan':'Milan',
    'Parma Calcio 1913':'Parma', 
    'Verona': 'Hellas Verona'

    #LIGUE 1 NONE
}
cols = ["homeTeamName", "awayTeamName"]

df_laliga[cols] = df_laliga[cols].replace(MAPPING_NAMES_WHOSCORED_TO_FOTMOB)

In [148]:
compare_team_names(laliga, "name_team_fotmob",df_laliga1, "homeTeamName")

Total únicos name_team_fotmob: 20
Total únicos homeTeamName: 20

Solo en name_team_fotmob:
None

Solo en homeTeamName:
None

Coinciden: 20


{'only_in_col1': set(),
 'only_in_col2': set(),
 'intersection': {'Athletic Club',
  'Atletico Madrid',
  'Barcelona',
  'Celta Vigo',
  'Deportivo Alaves',
  'Elche',
  'Espanyol',
  'Getafe',
  'Girona',
  'Levante',
  'Mallorca',
  'Osasuna',
  'Rayo Vallecano',
  'Real Betis',
  'Real Madrid',
  'Real Oviedo',
  'Real Sociedad',
  'Sevilla',
  'Valencia',
  'Villarreal'}}

In [149]:
compare_team_names(laliga, "name_team_fotmob",df_laliga1, "awayTeamName")

Total únicos name_team_fotmob: 20
Total únicos awayTeamName: 20

Solo en name_team_fotmob:
None

Solo en awayTeamName:
None

Coinciden: 20


{'only_in_col1': set(),
 'only_in_col2': set(),
 'intersection': {'Athletic Club',
  'Atletico Madrid',
  'Barcelona',
  'Celta Vigo',
  'Deportivo Alaves',
  'Elche',
  'Espanyol',
  'Getafe',
  'Girona',
  'Levante',
  'Mallorca',
  'Osasuna',
  'Rayo Vallecano',
  'Real Betis',
  'Real Madrid',
  'Real Oviedo',
  'Real Sociedad',
  'Sevilla',
  'Valencia',
  'Villarreal'}}

MATCHES 2025-2026 PREMIER LEAGUE

In [150]:
premierleague=df_all_teams_league_venues_photo[df_all_teams_league_venues_photo['name_league_fotmob']=='Premier League'].reset_index(drop=True)
premierleague

,name_team_fotmob,shortname_team_fotmob,id_team_fotmob,url_team_fotmob,logo_team_fotmob,season,id_league_fotmob,name_league_fotmob,country_code_fotmob,name_venue,city_venue,lat_venue,lon_venue,Surface,Capacity,Opened,url_photo_stadium
0,Arsenal,Arsenal,9825,https://www.fotmob.com/teams/9825/overview/ars...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Emirates Stadium,London,51.555074614,-0.108457804,Grass,60704,2006,https://cdn.resfu.com/img_data/estadios/origin...
1,Manchester City,Man City,8456,https://www.fotmob.com/teams/8456/overview/man...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Etihad Stadium,Manchester,53.483152259,-2.200409174,Grass,61470,2002,https://cdn.resfu.com/img_data/estadios/origin...
2,Manchester United,Man United,10260,https://www.fotmob.com/teams/10260/overview/ma...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Old Trafford,Manchester,53.463094458,-2.291373610,Grass,74244,1910,https://cdn.resfu.com/img_data/estadios/origin...
3,Aston Villa,Aston Villa,10252,https://www.fotmob.com/teams/10252/overview/as...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Villa Park,Birmingham,52.509090736,-1.884841919,Grass,43205,1897,https://cdn.resfu.com/img_data/estadios/origin...
4,Liverpool,Liverpool,8650,https://www.fotmob.com/teams/8650/overview/liv...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Anfield,Liverpool,53.430827885,-2.960852981,Grass,61276,1884,https://cdn.resfu.com/img_data/estadios/origin...
5,AFC Bournemouth,Bournemouth,8678,https://www.fotmob.com/teams/8678/overview/afc...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Vitality Stadium,"Bournemouth, Dorset",50.735240532,-1.838309616,Grass,11307,2001,https://cdn.resfu.com/img_data/estadios/origin...
6,Sunderland,Sunderland,8472,https://www.fotmob.com/teams/8472/overview/sun...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Stadium of Light,Sunderland,54.914421504,-1.388177276,Grass,49000,1997,https://cdn.resfu.com/img_data/estadios/origin...
7,Brighton & Hove Albion,Brighton,10204,https://www.fotmob.com/teams/10204/overview/br...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,American Express Stadium,"Falmer, East Sussex",50.861579551,-0.083642006,Grass,31876,2011,https://cdn.resfu.com/img_data/estadios/origin...
8,Brentford,Brentford,9937,https://www.fotmob.com/teams/9937/overview/bre...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Gtech Community Stadium,"Brentford, Middlesex",51.490729000,-0.288884000,Grass,17250,2020,https://cdn.resfu.com/img_data/estadios/origin...
9,Chelsea,Chelsea,8455,https://www.fotmob.com/teams/8455/overview/che...,https://images.fotmob.com/image_resources/logo...,2025/2026,47,Premier League,ENG,Stamford Bridge,London,51.481711963,-0.190974623,Grass,40044,1877,https://cdn.resfu.com/img_data/estadios/origin...


In [145]:
url2 = "https://www.whoscored.com/regions/252/tournaments/2/seasons/10743/stages/24533/fixtures/england-premier-league-2025-2026"
df_premier = scrape_whoscored(url2, headless=False)
df_premier

📋 Liga: England Premier League | Temporada: 2025-2026 | tournamentId: 2 | seasonId: 10743
🌐 Cargando: https://www.whoscored.com/regions/252/tournaments/2/seasons/10743/stages/24533/fixtures/england-premier-league-2025-2026
✅ Página cargada y partidos visibles en DOM
⏪ Navegando al primer mes...
  ← Apr 2026
  ← Mar 2026
  ← Feb 2026
  ← Jan 2026
  ← Dec 2025
  ← Nov 2025
  ← Oct 2025
  ← Sep 2025
  ← Aug 2025
✅ Primer mes alcanzado tras 9 clicks prev

📅 Extrayendo: Aug 2025
  ✓ 30 partidos

📅 Extrayendo: Sep 2025
  ✓ 30 partidos

📅 Extrayendo: Oct 2025
  ✓ 30 partidos

📅 Extrayendo: Nov 2025
  ✓ 40 partidos

📅 Extrayendo: Dec 2025
  ✓ 56 partidos

📅 Extrayendo: Jan 2026
  ✓ 49 partidos

📅 Extrayendo: Feb 2026
  ✓ 42 partidos

📅 Extrayendo: Mar 2026
  ✓ 32 partidos

📅 Extrayendo: Apr 2026
  ✓ 30 partidos

📅 Extrayendo: May 2026
  ✓ 41 partidos
🏁 DOM no cambió tras click next — fin del calendario

🎉 Total partidos: 380


,regionId,tournamentId,seasonId,stageId,leagueName,season,match_id,match_url,calendar_month,status,homeTeamName,awayTeamName,homeTeamId,awayTeamId,homeScore,awayScore,match_date,homeLogo,awayLogo
0,252,2,10743,24533,England Premier League,2025-2026,1903117,https://www.whoscored.com/matches/1903117/live...,Aug 2025,FT,Liverpool,Bournemouth,26,183,4,2,2025-08-15,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
1,252,2,10743,24533,England Premier League,2025-2026,1903118,https://www.whoscored.com/matches/1903118/live...,Aug 2025,FT,Aston Villa,Newcastle,24,23,0,0,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
2,252,2,10743,24533,England Premier League,2025-2026,1903119,https://www.whoscored.com/matches/1903119/live...,Aug 2025,FT,Brighton,Fulham,211,170,1,1,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
3,252,2,10743,24533,England Premier League,2025-2026,1903121,https://www.whoscored.com/matches/1903121/live...,Aug 2025,FT,Sunderland,West Ham,16,29,3,0,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
4,252,2,10743,24533,England Premier League,2025-2026,1903122,https://www.whoscored.com/matches/1903122/live...,Aug 2025,FT,Tottenham,Burnley,30,184,3,0,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,252,2,10743,24533,England Premier League,2025-2026,1903364,https://www.whoscored.com/matches/1903364/live...,May 2026,FT,Nottingham Forest,Bournemouth,174,183,1,1,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
376,252,2,10743,24533,England Premier League,2025-2026,1903367,https://www.whoscored.com/matches/1903367/live...,May 2026,FT,Sunderland,Chelsea,16,15,2,1,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
377,252,2,10743,24533,England Premier League,2025-2026,1903370,https://www.whoscored.com/matches/1903370/live...,May 2026,FT,Tottenham,Everton,30,31,1,0,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
378,252,2,10743,24533,England Premier League,2025-2026,1903373,https://www.whoscored.com/matches/1903373/live...,May 2026,FT,West Ham,Leeds,29,19,3,0,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...


In [167]:
df_premier1= transformation_fixtures_matchweek(df_premier)
df_premier1

,regionId,tournamentId,seasonId,stageId,leagueName,season,match_id,match_url,calendar_month,status,homeTeamName,awayTeamName,homeTeamId,awayTeamId,homeScore,awayScore,match_date,homeLogo,awayLogo,matchweek_id,matchround
0,252,2,10743,24533,England Premier League,2025-2026,1903117,https://www.whoscored.com/matches/1903117/live...,Aug 2025,FT,Liverpool,AFC Bournemouth,26,183,4,2,2025-08-15,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
1,252,2,10743,24533,England Premier League,2025-2026,1903118,https://www.whoscored.com/matches/1903118/live...,Aug 2025,FT,Aston Villa,Newcastle United,24,23,0,0,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
2,252,2,10743,24533,England Premier League,2025-2026,1903119,https://www.whoscored.com/matches/1903119/live...,Aug 2025,FT,Brighton & Hove Albion,Fulham,211,170,1,1,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
3,252,2,10743,24533,England Premier League,2025-2026,1903121,https://www.whoscored.com/matches/1903121/live...,Aug 2025,FT,Sunderland,West Ham United,16,29,3,0,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
4,252,2,10743,24533,England Premier League,2025-2026,1903122,https://www.whoscored.com/matches/1903122/live...,Aug 2025,FT,Tottenham Hotspur,Burnley,30,184,3,0,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,252,2,10743,24533,England Premier League,2025-2026,1903364,https://www.whoscored.com/matches/1903364/live...,May 2026,FT,Nottingham Forest,AFC Bournemouth,174,183,1,1,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
376,252,2,10743,24533,England Premier League,2025-2026,1903367,https://www.whoscored.com/matches/1903367/live...,May 2026,FT,Sunderland,Chelsea,16,15,2,1,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
377,252,2,10743,24533,England Premier League,2025-2026,1903370,https://www.whoscored.com/matches/1903370/live...,May 2026,FT,Tottenham Hotspur,Everton,30,31,1,0,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
378,252,2,10743,24533,England Premier League,2025-2026,1903373,https://www.whoscored.com/matches/1903373/live...,May 2026,FT,West Ham United,Leeds United,29,19,3,0,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38


In [169]:
df_premier1[df_premier1["matchweek_id"].isna()][
    ["match_id", "match_date", "homeTeamName", "awayTeamName"]
]

,match_id,match_date,homeTeamName,awayTeamName


In [170]:
cols = ["homeTeamName", "awayTeamName"]

df_premier1[cols] = df_premier1[cols].replace(MAPPING_NAMES_WHOSCORED_TO_FOTMOB)

In [171]:
compare_team_names(premierleague, "name_team_fotmob",df_premier1, "homeTeamName")

Total únicos name_team_fotmob: 20
Total únicos homeTeamName: 20

Solo en name_team_fotmob:
None

Solo en homeTeamName:
None

Coinciden: 20


{'only_in_col1': set(),
 'only_in_col2': set(),
 'intersection': {'AFC Bournemouth',
  'Arsenal',
  'Aston Villa',
  'Brentford',
  'Brighton & Hove Albion',
  'Burnley',
  'Chelsea',
  'Crystal Palace',
  'Everton',
  'Fulham',
  'Leeds United',
  'Liverpool',
  'Manchester City',
  'Manchester United',
  'Newcastle United',
  'Nottingham Forest',
  'Sunderland',
  'Tottenham Hotspur',
  'West Ham United',
  'Wolverhampton Wanderers'}}

In [173]:
compare_team_names(premierleague, "name_team_fotmob",df_premier1, "awayTeamName")

Total únicos name_team_fotmob: 20
Total únicos awayTeamName: 20

Solo en name_team_fotmob:
None

Solo en awayTeamName:
None

Coinciden: 20


{'only_in_col1': set(),
 'only_in_col2': set(),
 'intersection': {'AFC Bournemouth',
  'Arsenal',
  'Aston Villa',
  'Brentford',
  'Brighton & Hove Albion',
  'Burnley',
  'Chelsea',
  'Crystal Palace',
  'Everton',
  'Fulham',
  'Leeds United',
  'Liverpool',
  'Manchester City',
  'Manchester United',
  'Newcastle United',
  'Nottingham Forest',
  'Sunderland',
  'Tottenham Hotspur',
  'West Ham United',
  'Wolverhampton Wanderers'}}

MATCHES 2025-2026 BUNDESLIGA

In [154]:
bundesliga=df_all_teams_league_venues_photo[df_all_teams_league_venues_photo['name_league_fotmob']=='Bundesliga'].reset_index(drop=True)
bundesliga

,name_team_fotmob,shortname_team_fotmob,id_team_fotmob,url_team_fotmob,logo_team_fotmob,season,id_league_fotmob,name_league_fotmob,country_code_fotmob,name_venue,city_venue,lat_venue,lon_venue,Surface,Capacity,Opened,url_photo_stadium
0,Bayern München,Bayern München,9823,https://www.fotmob.com/teams/9823/overview/bay...,https://images.fotmob.com/image_resources/logo...,2025/2026,54,Bundesliga,GER,Allianz Arena,München,48.218779054,11.624743491,Grass,75024,2005,https://cdn.resfu.com/img_data/estadios/origin...
1,Borussia Dortmund,Dortmund,9789,https://www.fotmob.com/teams/9789/overview/bor...,https://images.fotmob.com/image_resources/logo...,2025/2026,54,Bundesliga,GER,SIGNAL IDUNA PARK,Dortmund,51.492606578,7.451777458,Grass,81365,1974,https://cdn.resfu.com/img_data/estadios/origin...
2,RB Leipzig,RB Leipzig,178475,https://www.fotmob.com/teams/178475/overview/r...,https://images.fotmob.com/image_resources/logo...,2025/2026,54,Bundesliga,GER,Red Bull Arena,Leipzig,51.345776141,12.348289490,Grass,47800,2004,https://cdn.resfu.com/img_data/estadios/origin...
3,VfB Stuttgart,VfB Stuttgart,10269,https://www.fotmob.com/teams/10269/overview/vf...,https://images.fotmob.com/image_resources/logo...,2025/2026,54,Bundesliga,GER,MHPArena,Stuttgart,48.792284176,9.232023954,Grass,60469,1933,https://cdn.resfu.com/img_data/estadios/origin...
4,Hoffenheim,Hoffenheim,8226,https://www.fotmob.com/teams/8226/overview/hof...,https://images.fotmob.com/image_resources/logo...,2025/2026,54,Bundesliga,GER,PreZero Arena,Sinsheim,49.238005290,8.887740970,Grass,30164,2009,https://cdn.resfu.com/img_data/estadios/origin...
5,Bayer Leverkusen,Leverkusen,8178,https://www.fotmob.com/teams/8178/overview/bay...,https://images.fotmob.com/image_resources/logo...,2025/2026,54,Bundesliga,GER,BayArena,Leverkusen,51.038213121,7.002249956,Grass,30210,1958,https://cdn.resfu.com/img_data/estadios/origin...
6,Freiburg,Freiburg,8358,https://www.fotmob.com/teams/8358/overview/fre...,https://images.fotmob.com/image_resources/logo...,2025/2026,54,Bundesliga,GER,Europa-Park Stadion,Freiburg im Breisgau,48.021945037,7.830065164,Grass,34700,2021,https://cdn.resfu.com/img_data/estadios/origin...
7,Eintracht Frankfurt,Frankfurt,9810,https://www.fotmob.com/teams/9810/overview/ein...,https://images.fotmob.com/image_resources/logo...,2025/2026,54,Bundesliga,GER,Deutsche Bank Park,Frankfurt am Main,50.068577360,8.645448312,Grass,59500,2005,https://cdn.resfu.com/img_data/estadios/origin...
8,Augsburg,Augsburg,8406,https://www.fotmob.com/teams/8406/overview/aug...,https://images.fotmob.com/image_resources/logo...,2025/2026,54,Bundesliga,GER,WWK Arena,Augsburg,48.323101360,10.885949135,Grass,30662,2009,https://cdn.resfu.com/img_data/estadios/origin...
9,Mainz 05,Mainz,9905,https://www.fotmob.com/teams/9905/overview/mai...,https://images.fotmob.com/image_resources/logo...,2025/2026,54,Bundesliga,GER,MEWA ARENA,Mainz,49.984627000,8.224125000,Grass,34034,2011,https://cdn.resfu.com/img_data/estadios/origin...


In [129]:
url3= "https://www.whoscored.com/regions/81/tournaments/3/seasons/10720/stages/24478/fixtures/germany-bundesliga-2025-2026"
df_bundesliga = scrape_whoscored(url3, headless=False)
df_bundesliga

📋 Liga: Germany Bundesliga | Temporada: 2025-2026 | tournamentId: 3 | seasonId: 10720
🌐 Cargando: https://www.whoscored.com/regions/81/tournaments/3/seasons/10720/stages/24478/fixtures/germany-bundesliga-2025-2026
✅ Página cargada y partidos visibles en DOM
⏪ Navegando al primer mes...
  ← Apr 2026
  ← Mar 2026
  ← Feb 2026
  ← Jan 2026
  ← Dec 2025
  ← Nov 2025
  ← Oct 2025
  ← Sep 2025
  ← Aug 2025
✅ Primer mes alcanzado tras 9 clicks prev

📅 Extrayendo: Aug 2025
  ✓ 18 partidos

📅 Extrayendo: Sep 2025
  ✓ 27 partidos

📅 Extrayendo: Oct 2025
  ✓ 28 partidos

📅 Extrayendo: Nov 2025
  ✓ 35 partidos

📅 Extrayendo: Dec 2025
  ✓ 27 partidos

📅 Extrayendo: Jan 2026
  ✓ 42 partidos

📅 Extrayendo: Feb 2026
  ✓ 35 partidos

📅 Extrayendo: Mar 2026
  ✓ 31 partidos

📅 Extrayendo: Apr 2026
  ✓ 36 partidos

📅 Extrayendo: May 2026
  ✓ 27 partidos
🏁 DOM no cambió tras click next — fin del calendario

🎉 Total partidos: 306


,regionId,tournamentId,seasonId,stageId,leagueName,season,match_id,match_url,calendar_month,status,homeTeamName,awayTeamName,homeTeamId,awayTeamId,homeScore,awayScore,match_date,homeLogo,awayLogo
0,81,3,10720,24478,Germany Bundesliga,2025-2026,1908319,https://www.whoscored.com/matches/1908319/live...,Aug 2025,FT,Bayern Munich,RB Leipzig,37,7614,6,0,2025-08-22,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
1,81,3,10720,24478,Germany Bundesliga,2025-2026,1910600,https://www.whoscored.com/matches/1910600/live...,Aug 2025,FT,FC Heidenheim,Wolfsburg,4852,33,1,3,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
2,81,3,10720,24478,Germany Bundesliga,2025-2026,1910601,https://www.whoscored.com/matches/1910601/live...,Aug 2025,FT,Bayer Leverkusen,Hoffenheim,36,1211,1,2,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
3,81,3,10720,24478,Germany Bundesliga,2025-2026,1910603,https://www.whoscored.com/matches/1910603/live...,Aug 2025,FT,Union Berlin,VfB Stuttgart,796,41,2,1,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
4,81,3,10720,24478,Germany Bundesliga,2025-2026,1910604,https://www.whoscored.com/matches/1910604/live...,Aug 2025,FT,Eintracht Frankfurt,Werder Bremen,45,42,4,1,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,81,3,10720,24478,Germany Bundesliga,2025-2026,1910898,https://www.whoscored.com/matches/1910898/live...,May 2026,FT,Freiburg,RB Leipzig,50,7614,4,1,2026-05-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
302,81,3,10720,24478,Germany Bundesliga,2025-2026,1910899,https://www.whoscored.com/matches/1910899/live...,May 2026,FT,FC Heidenheim,Mainz 05,4852,219,0,2,2026-05-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
303,81,3,10720,24478,Germany Bundesliga,2025-2026,1910900,https://www.whoscored.com/matches/1910900/live...,May 2026,FT,St. Pauli,Wolfsburg,283,33,1,3,2026-05-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
304,81,3,10720,24478,Germany Bundesliga,2025-2026,1910901,https://www.whoscored.com/matches/1910901/live...,May 2026,FT,Union Berlin,Augsburg,796,1730,4,0,2026-05-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...


In [155]:
df_bundesliga1= transformation_fixtures_matchweek(df_bundesliga)
df_bundesliga1

,regionId,tournamentId,seasonId,stageId,leagueName,season,match_id,match_url,calendar_month,status,homeTeamName,awayTeamName,homeTeamId,awayTeamId,homeScore,awayScore,match_date,homeLogo,awayLogo,matchweek_id,matchround
0,81,3,10720,24478,Germany Bundesliga,2025-2026,1908319,https://www.whoscored.com/matches/1908319/live...,Aug 2025,FT,Bayern Munich,RB Leipzig,37,7614,6,0,2025-08-22,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
1,81,3,10720,24478,Germany Bundesliga,2025-2026,1910600,https://www.whoscored.com/matches/1910600/live...,Aug 2025,FT,FC Heidenheim,Wolfsburg,4852,33,1,3,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
2,81,3,10720,24478,Germany Bundesliga,2025-2026,1910601,https://www.whoscored.com/matches/1910601/live...,Aug 2025,FT,Bayer Leverkusen,Hoffenheim,36,1211,1,2,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
3,81,3,10720,24478,Germany Bundesliga,2025-2026,1910603,https://www.whoscored.com/matches/1910603/live...,Aug 2025,FT,Union Berlin,VfB Stuttgart,796,41,2,1,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
4,81,3,10720,24478,Germany Bundesliga,2025-2026,1910604,https://www.whoscored.com/matches/1910604/live...,Aug 2025,FT,Eintracht Frankfurt,Werder Bremen,45,42,4,1,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,81,3,10720,24478,Germany Bundesliga,2025-2026,1910898,https://www.whoscored.com/matches/1910898/live...,May 2026,FT,Freiburg,RB Leipzig,50,7614,4,1,2026-05-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,34,34
302,81,3,10720,24478,Germany Bundesliga,2025-2026,1910899,https://www.whoscored.com/matches/1910899/live...,May 2026,FT,FC Heidenheim,Mainz 05,4852,219,0,2,2026-05-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,34,34
303,81,3,10720,24478,Germany Bundesliga,2025-2026,1910900,https://www.whoscored.com/matches/1910900/live...,May 2026,FT,St. Pauli,Wolfsburg,283,33,1,3,2026-05-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,34,34
304,81,3,10720,24478,Germany Bundesliga,2025-2026,1910901,https://www.whoscored.com/matches/1910901/live...,May 2026,FT,Union Berlin,Augsburg,796,1730,4,0,2026-05-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,34,34


In [156]:
df_bundesliga1['matchweek_id'].value_counts().sort_index()

matchweek_id
1     9
2     9
3     9
4     9
5     9
6     9
7     9
8     9
9     9
10    9
11    9
12    9
13    9
14    9
15    9
16    9
17    9
18    9
19    9
20    9
21    9
22    9
23    9
24    9
25    9
26    9
28    9
29    9
30    9
31    9
32    9
33    9
34    9
Name: count, dtype: Int64

In [157]:
cols = ["homeTeamName", "awayTeamName"]

df_bundesliga1[cols] = df_bundesliga1[cols].replace(MAPPING_NAMES_WHOSCORED_TO_FOTMOB)

In [158]:
compare_team_names(bundesliga, "name_team_fotmob",df_bundesliga1, "homeTeamName")

Total únicos name_team_fotmob: 18
Total únicos homeTeamName: 18

Solo en name_team_fotmob:
None

Solo en homeTeamName:
None

Coinciden: 18


{'only_in_col1': set(),
 'only_in_col2': set(),
 'intersection': {'1. FC Köln',
  'Augsburg',
  'Bayer Leverkusen',
  'Bayern München',
  'Borussia Dortmund',
  'Borussia Mönchengladbach',
  'Eintracht Frankfurt',
  'FC Heidenheim',
  'Freiburg',
  'Hamburger SV',
  'Hoffenheim',
  'Mainz 05',
  'RB Leipzig',
  'St. Pauli',
  'Union Berlin',
  'VfB Stuttgart',
  'Werder Bremen',
  'Wolfsburg'}}

In [160]:
compare_team_names(bundesliga, "name_team_fotmob",df_bundesliga1, "awayTeamName")

Total únicos name_team_fotmob: 18
Total únicos awayTeamName: 18

Solo en name_team_fotmob:
None

Solo en awayTeamName:
None

Coinciden: 18


{'only_in_col1': set(),
 'only_in_col2': set(),
 'intersection': {'1. FC Köln',
  'Augsburg',
  'Bayer Leverkusen',
  'Bayern München',
  'Borussia Dortmund',
  'Borussia Mönchengladbach',
  'Eintracht Frankfurt',
  'FC Heidenheim',
  'Freiburg',
  'Hamburger SV',
  'Hoffenheim',
  'Mainz 05',
  'RB Leipzig',
  'St. Pauli',
  'Union Berlin',
  'VfB Stuttgart',
  'Werder Bremen',
  'Wolfsburg'}}

MATCHES 2025-2026 SERIE A

In [174]:
seriea=df_all_teams_league_venues_photo[df_all_teams_league_venues_photo['name_league_fotmob']=='Serie A'].reset_index(drop=True)
seriea

,name_team_fotmob,shortname_team_fotmob,id_team_fotmob,url_team_fotmob,logo_team_fotmob,season,id_league_fotmob,name_league_fotmob,country_code_fotmob,name_venue,city_venue,lat_venue,lon_venue,Surface,Capacity,Opened,url_photo_stadium
0,Inter,Inter,8636,https://www.fotmob.com/teams/8636/overview/inter,https://images.fotmob.com/image_resources/logo...,2025/2026,55,Serie A,ITA,Stadio Giuseppe Meazza,Milano,45.478083998,9.123979211,Grass,75817,1926,https://cdn.resfu.com/img_data/estadios/origin...
1,Napoli,Napoli,9875,https://www.fotmob.com/teams/9875/overview/napoli,https://images.fotmob.com/image_resources/logo...,2025/2026,55,Serie A,ITA,Stadio Diego Armando Maradona,Napoli,40.827974049,14.192930460,Grass,54726,1959,https://cdn.resfu.com/img_data/estadios/origin...
2,Roma,Roma,8686,https://www.fotmob.com/teams/8686/overview/roma,https://images.fotmob.com/image_resources/logo...,2025/2026,55,Serie A,ITA,Stadio Olimpico,Roma,41.933947912,12.454757094,Grass,70634,1932,https://cdn.resfu.com/img_data/estadios/origin...
3,Como,Como,10171,https://www.fotmob.com/teams/10171/overview/como,https://images.fotmob.com/image_resources/logo...,2025/2026,55,Serie A,ITA,Stadio Giuseppe Sinigaglia,Como,45.813856655,9.072376192,Grass,13602,1927,https://cdn.resfu.com/img_data/estadios/origin...
4,Milan,Milan,8564,https://www.fotmob.com/teams/8564/overview/milan,https://images.fotmob.com/image_resources/logo...,2025/2026,55,Serie A,ITA,Stadio Giuseppe Meazza,Milano,45.478083998,9.123979211,Grass,75817,1926,https://cdn.resfu.com/img_data/estadios/origin...
5,Juventus,Juventus,9885,https://www.fotmob.com/teams/9885/overview/juv...,https://images.fotmob.com/image_resources/logo...,2025/2026,55,Serie A,ITA,Allianz Stadium,Torino,45.109527284,7.641302313,Grass,41689,2011,https://cdn.resfu.com/img_data/estadios/origin...
6,Atalanta,Atalanta,8524,https://www.fotmob.com/teams/8524/overview/ata...,https://images.fotmob.com/image_resources/logo...,2025/2026,55,Serie A,ITA,New Balance Arena,Bergamo,45.709236083,9.680693150,Grass,24950,1928,https://cdn.resfu.com/img_data/estadios/origin...
7,Bologna,Bologna,9857,https://www.fotmob.com/teams/9857/overview/bol...,https://images.fotmob.com/image_resources/logo...,2025/2026,55,Serie A,ITA,Stadio Renato Dall'Ara,Bologna,44.492290396,11.309956759,Grass,39279,1927,https://cdn.resfu.com/img_data/estadios/origin...
8,Lazio,Lazio,8543,https://www.fotmob.com/teams/8543/overview/lazio,https://images.fotmob.com/image_resources/logo...,2025/2026,55,Serie A,ITA,Stadio Olimpico,Roma,41.933947912,12.454757094,Grass,70634,1932,https://cdn.resfu.com/img_data/estadios/origin...
9,Udinese,Udinese,8600,https://www.fotmob.com/teams/8600/overview/udi...,https://images.fotmob.com/image_resources/logo...,2025/2026,55,Serie A,ITA,Bluenergy Stadium,Udine,46.081644302,13.200062513,Grass,25952,1976,https://cdn.resfu.com/img_data/estadios/origin...


In [175]:
url4= "https://www.whoscored.com/regions/108/tournaments/5/seasons/10732/stages/24500/fixtures/italy-serie-a-2025-2026"
df_seriea = scrape_whoscored(url4, headless=False)
df_seriea

📋 Liga: Italy Serie A | Temporada: 2025-2026 | tournamentId: 5 | seasonId: 10732
🌐 Cargando: https://www.whoscored.com/regions/108/tournaments/5/seasons/10732/stages/24500/fixtures/italy-serie-a-2025-2026
✅ Página cargada y partidos visibles en DOM
⏪ Navegando al primer mes...
  ← Apr 2026
  ← Mar 2026
  ← Feb 2026
  ← Jan 2026
  ← Dec 2025
  ← Nov 2025
  ← Oct 2025
  ← Sep 2025
  ← Aug 2025
✅ Primer mes alcanzado tras 9 clicks prev

📅 Extrayendo: Aug 2025
  ✓ 20 partidos

📅 Extrayendo: Sep 2025
  ✓ 30 partidos

📅 Extrayendo: Oct 2025
  ✓ 40 partidos

📅 Extrayendo: Nov 2025
  ✓ 39 partidos

📅 Extrayendo: Dec 2025
  ✓ 37 partidos

📅 Extrayendo: Jan 2026
  ✓ 58 partidos

📅 Extrayendo: Feb 2026
  ✓ 40 partidos

📅 Extrayendo: Mar 2026
  ✓ 36 partidos

📅 Extrayendo: Apr 2026
  ✓ 40 partidos

📅 Extrayendo: May 2026
  ✓ 40 partidos
🏁 DOM no cambió tras click next — fin del calendario

🎉 Total partidos: 380


,regionId,tournamentId,seasonId,stageId,leagueName,season,match_id,match_url,calendar_month,status,homeTeamName,awayTeamName,homeTeamId,awayTeamId,homeScore,awayScore,match_date,homeLogo,awayLogo
0,108,5,10732,24500,Italy Serie A,2025-2026,1901064,https://www.whoscored.com/matches/1901064/live...,Aug 2025,FT,Genoa,Lecce,278,79,0,0,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
1,108,5,10732,24500,Italy Serie A,2025-2026,1901069,https://www.whoscored.com/matches/1901069/live...,Aug 2025,FT,Sassuolo,Napoli,2889,276,0,2,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
2,108,5,10732,24500,Italy Serie A,2025-2026,1901067,https://www.whoscored.com/matches/1901067/live...,Aug 2025,FT,AC Milan,Cremonese,80,2731,1,2,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
3,108,5,10732,24500,Italy Serie A,2025-2026,1901068,https://www.whoscored.com/matches/1901068/live...,Aug 2025,FT,Roma,Bologna,84,71,1,0,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
4,108,5,10732,24500,Italy Serie A,2025-2026,1901062,https://www.whoscored.com/matches/1901062/live...,Aug 2025,FT,Cagliari,Fiorentina,78,73,1,1,2025-08-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,108,5,10732,24500,Italy Serie A,2025-2026,1901440,https://www.whoscored.com/matches/1901440/live...,May 2026,FT,Verona,Roma,76,84,0,2,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
376,108,5,10732,24500,Italy Serie A,2025-2026,1901435,https://www.whoscored.com/matches/1901435/live...,May 2026,FT,Lecce,Genoa,79,278,1,0,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
377,108,5,10732,24500,Italy Serie A,2025-2026,1901436,https://www.whoscored.com/matches/1901436/live...,May 2026,FT,AC Milan,Cagliari,80,78,1,2,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
378,108,5,10732,24500,Italy Serie A,2025-2026,1901432,https://www.whoscored.com/matches/1901432/live...,May 2026,FT,Cremonese,Como,2731,1290,1,4,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...


In [185]:
df_seriea1= transformation_fixtures_matchweek(df_seriea)
df_seriea1

,regionId,tournamentId,seasonId,stageId,leagueName,season,match_id,match_url,calendar_month,status,homeTeamName,awayTeamName,homeTeamId,awayTeamId,homeScore,awayScore,match_date,homeLogo,awayLogo,matchweek_id,matchround
0,108,5,10732,24500,Italy Serie A,2025-2026,1901064,https://www.whoscored.com/matches/1901064/live...,Aug 2025,FT,Genoa,Lecce,278,79,0,0,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
1,108,5,10732,24500,Italy Serie A,2025-2026,1901069,https://www.whoscored.com/matches/1901069/live...,Aug 2025,FT,Sassuolo,Napoli,2889,276,0,2,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
2,108,5,10732,24500,Italy Serie A,2025-2026,1901067,https://www.whoscored.com/matches/1901067/live...,Aug 2025,FT,AC Milan,Cremonese,80,2731,1,2,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
3,108,5,10732,24500,Italy Serie A,2025-2026,1901068,https://www.whoscored.com/matches/1901068/live...,Aug 2025,FT,Roma,Bologna,84,71,1,0,2025-08-23,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
4,108,5,10732,24500,Italy Serie A,2025-2026,1901062,https://www.whoscored.com/matches/1901062/live...,Aug 2025,FT,Cagliari,Fiorentina,78,73,1,1,2025-08-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,108,5,10732,24500,Italy Serie A,2025-2026,1901440,https://www.whoscored.com/matches/1901440/live...,May 2026,FT,Verona,Roma,76,84,0,2,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
376,108,5,10732,24500,Italy Serie A,2025-2026,1901435,https://www.whoscored.com/matches/1901435/live...,May 2026,FT,Lecce,Genoa,79,278,1,0,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
377,108,5,10732,24500,Italy Serie A,2025-2026,1901436,https://www.whoscored.com/matches/1901436/live...,May 2026,FT,AC Milan,Cagliari,80,78,1,2,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38
378,108,5,10732,24500,Italy Serie A,2025-2026,1901432,https://www.whoscored.com/matches/1901432/live...,May 2026,FT,Cremonese,Como,2731,1290,1,4,2026-05-24,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,38,38


In [188]:
cols = ["homeTeamName", "awayTeamName"]
df_seriea1[cols] = df_seriea1[cols].replace(MAPPING_NAMES_WHOSCORED_TO_FOTMOB)

In [189]:
compare_team_names(seriea, "name_team_fotmob",df_seriea1, "homeTeamName")

Total únicos name_team_fotmob: 20
Total únicos homeTeamName: 20

Solo en name_team_fotmob:
None

Solo en homeTeamName:
None

Coinciden: 20


{'only_in_col1': set(),
 'only_in_col2': set(),
 'intersection': {'Atalanta',
  'Bologna',
  'Cagliari',
  'Como',
  'Cremonese',
  'Fiorentina',
  'Genoa',
  'Hellas Verona',
  'Inter',
  'Juventus',
  'Lazio',
  'Lecce',
  'Milan',
  'Napoli',
  'Parma',
  'Pisa',
  'Roma',
  'Sassuolo',
  'Torino',
  'Udinese'}}

In [190]:
compare_team_names(seriea, "name_team_fotmob",df_seriea1, "awayTeamName")

Total únicos name_team_fotmob: 20
Total únicos awayTeamName: 20

Solo en name_team_fotmob:
None

Solo en awayTeamName:
None

Coinciden: 20


{'only_in_col1': set(),
 'only_in_col2': set(),
 'intersection': {'Atalanta',
  'Bologna',
  'Cagliari',
  'Como',
  'Cremonese',
  'Fiorentina',
  'Genoa',
  'Hellas Verona',
  'Inter',
  'Juventus',
  'Lazio',
  'Lecce',
  'Milan',
  'Napoli',
  'Parma',
  'Pisa',
  'Roma',
  'Sassuolo',
  'Torino',
  'Udinese'}}

MATCHES 2025-2026 LIGUE 1

In [206]:
ligue1=df_all_teams_league_venues_photo[df_all_teams_league_venues_photo['name_league_fotmob']=='Ligue 1'].reset_index(drop=True)
ligue1

,name_team_fotmob,shortname_team_fotmob,id_team_fotmob,url_team_fotmob,logo_team_fotmob,season,id_league_fotmob,name_league_fotmob,country_code_fotmob,name_venue,city_venue,lat_venue,lon_venue,Surface,Capacity,Opened,url_photo_stadium
0,Paris Saint-Germain,PSG,9847,https://www.fotmob.com/teams/9847/overview/par...,https://images.fotmob.com/image_resources/logo...,2025/2026,53,Ligue 1,FRA,Parc des Princes,Paris,48.841433445,2.253048867,Grass,47929,1897,https://cdn.resfu.com/img_data/estadios/origin...
1,Lens,Lens,8588,https://www.fotmob.com/teams/8588/overview/lens,https://images.fotmob.com/image_resources/logo...,2025/2026,53,Ligue 1,FRA,Stade Bollaert-Delelis,Lens,50.432866762,2.814940810,Grass,41233,1933,https://cdn.resfu.com/img_data/estadios/origin...
2,Lille,Lille,8639,https://www.fotmob.com/teams/8639/overview/lille,https://images.fotmob.com/image_resources/logo...,2025/2026,53,Ligue 1,FRA,Decathlon Arena - Stade Pierre-Mauroy,Villeneuve d'Ascq,50.611965136,3.130483910,Grass,50083,2012,https://cdn.resfu.com/img_data/estadios/origin...
3,Lyon,Lyon,9748,https://www.fotmob.com/teams/9748/overview/lyon,https://images.fotmob.com/image_resources/logo...,2025/2026,53,Ligue 1,FRA,Groupama Stadium,Décines-Charpieu,45.765310356,4.982063884,Grass,61556,2016,https://cdn.resfu.com/img_data/estadios/origin...
4,Marseille,Marseille,8592,https://www.fotmob.com/teams/8592/overview/mar...,https://images.fotmob.com/image_resources/logo...,2025/2026,53,Ligue 1,FRA,Orange Vélodrome,Marseille,43.269870275,5.395853519,Grass,67394,1937,https://cdn.resfu.com/img_data/estadios/origin...
5,Rennes,Rennes,9851,https://www.fotmob.com/teams/9851/overview/rennes,https://images.fotmob.com/image_resources/logo...,2025/2026,53,Ligue 1,FRA,Roazhon Park,Rennes,48.107467009,-1.712884158,Grass,31127,1912,https://cdn.resfu.com/img_data/estadios/origin...
6,Monaco,Monaco,9829,https://www.fotmob.com/teams/9829/overview/monaco,https://images.fotmob.com/image_resources/logo...,2025/2026,53,Ligue 1,FRA,Stade Louis-II,Monaco,43.727563789,7.415542156,Grass,18523,1985,https://cdn.resfu.com/img_data/estadios/origin...
7,Strasbourg,Strasbourg,9848,https://www.fotmob.com/teams/9848/overview/str...,https://images.fotmob.com/image_resources/logo...,2025/2026,53,Ligue 1,FRA,Stade de la Meinau,Strasbourg,48.560068724,7.755156755,Grass,32047,1906,https://cdn.resfu.com/img_data/estadios/origin...
8,Toulouse,Toulouse,9941,https://www.fotmob.com/teams/9941/overview/tou...,https://images.fotmob.com/image_resources/logo...,2025/2026,53,Ligue 1,FRA,Stadium de Toulouse,Toulouse,43.583289762,1.434037685,Grass,33150,1937,https://cdn.resfu.com/img_data/estadios/origin...
9,Lorient,Lorient,8689,https://www.fotmob.com/teams/8689/overview/lor...,https://images.fotmob.com/image_resources/logo...,2025/2026,53,Ligue 1,FRA,Stade du Moustoir - Yves Allainmat,Lorient,47.748821296,-3.369020820,Artificial turf,18970,1959,https://cdn.resfu.com/img_data/estadios/origin...


In [187]:
url5= "https://www.whoscored.com/regions/74/tournaments/22/seasons/10792/stages/24609/fixtures/france-ligue-1-2025-2026"
df_ligue1 = scrape_whoscored(url5, headless=False)
df_ligue1

📋 Liga: France Ligue 1 | Temporada: 2025-2026 | tournamentId: 22 | seasonId: 10792
🌐 Cargando: https://www.whoscored.com/regions/74/tournaments/22/seasons/10792/stages/24609/fixtures/france-ligue-1-2025-2026
✅ Página cargada y partidos visibles en DOM
⏪ Navegando al primer mes...
  ← Apr 2026
  ← Mar 2026
  ← Feb 2026
  ← Jan 2026
  ← Dec 2025
  ← Nov 2025
  ← Oct 2025
  ← Sep 2025
  ← Aug 2025
✅ Primer mes alcanzado tras 9 clicks prev

📅 Extrayendo: Aug 2025
  ✓ 27 partidos

📅 Extrayendo: Sep 2025
  ✓ 27 partidos

📅 Extrayendo: Oct 2025
  ✓ 36 partidos

📅 Extrayendo: Nov 2025
  ✓ 36 partidos

📅 Extrayendo: Dec 2025
  ✓ 18 partidos

📅 Extrayendo: Jan 2026
  ✓ 31 partidos

📅 Extrayendo: Feb 2026
  ✓ 36 partidos

📅 Extrayendo: Mar 2026
  ✓ 31 partidos

📅 Extrayendo: Apr 2026
  ✓ 35 partidos

📅 Extrayendo: May 2026
  ✓ 29 partidos
🏁 DOM no cambió tras click next — fin del calendario

🎉 Total partidos: 306


,regionId,tournamentId,seasonId,stageId,leagueName,season,match_id,match_url,calendar_month,status,homeTeamName,awayTeamName,homeTeamId,awayTeamId,homeScore,awayScore,match_date,homeLogo,awayLogo
0,74,22,10792,24609,France Ligue 1,2025-2026,1911273,https://www.whoscored.com/matches/1911273/live...,Aug 2025,FT,Rennes,Marseille,313,249,1,0,2025-08-15,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
1,74,22,10792,24609,France Ligue 1,2025-2026,1911284,https://www.whoscored.com/matches/1911284/live...,Aug 2025,FT,Lens,Lyon,309,228,0,1,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
2,74,22,10792,24609,France Ligue 1,2025-2026,1911290,https://www.whoscored.com/matches/1911290/live...,Aug 2025,FT,Monaco,Le Havre,248,217,3,1,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
3,74,22,10792,24609,France Ligue 1,2025-2026,1911296,https://www.whoscored.com/matches/1911296/live...,Aug 2025,FT,Nice,Toulouse,613,246,0,1,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
4,74,22,10792,24609,France Ligue 1,2025-2026,1911281,https://www.whoscored.com/matches/1911281/live...,Aug 2025,FT,Brest,Lille,2332,607,3,3,2025-08-17,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,74,22,10792,24609,France Ligue 1,2025-2026,1911524,https://www.whoscored.com/matches/1911524/live...,May 2026,FT,Marseille,Rennes,249,313,3,1,2026-05-17,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
302,74,22,10792,24609,France Ligue 1,2025-2026,1911525,https://www.whoscored.com/matches/1911525/live...,May 2026,FT,Nantes,Toulouse,302,246,0,0,2026-05-17,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
303,74,22,10792,24609,France Ligue 1,2025-2026,1911526,https://www.whoscored.com/matches/1911526/live...,May 2026,FT,Nice,Metz,613,314,0,0,2026-05-17,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...
304,74,22,10792,24609,France Ligue 1,2025-2026,1911527,https://www.whoscored.com/matches/1911527/live...,May 2026,FT,Paris FC,Paris Saint-Germain,2832,304,2,1,2026-05-17,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...


In [201]:
df_ligue1= transformation_fixtures_matchweek(df_ligue1)
df_ligue1

,regionId,tournamentId,seasonId,stageId,leagueName,season,match_id,match_url,calendar_month,status,homeTeamName,awayTeamName,homeTeamId,awayTeamId,homeScore,awayScore,match_date,homeLogo,awayLogo,matchweek_id,matchround
0,74,22,10792,24609,France Ligue 1,2025-2026,1911273,https://www.whoscored.com/matches/1911273/live...,Aug 2025,FT,Rennes,Marseille,313,249,1,0,2025-08-15,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
1,74,22,10792,24609,France Ligue 1,2025-2026,1911284,https://www.whoscored.com/matches/1911284/live...,Aug 2025,FT,Lens,Lyon,309,228,0,1,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
2,74,22,10792,24609,France Ligue 1,2025-2026,1911290,https://www.whoscored.com/matches/1911290/live...,Aug 2025,FT,Monaco,Le Havre,248,217,3,1,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
3,74,22,10792,24609,France Ligue 1,2025-2026,1911296,https://www.whoscored.com/matches/1911296/live...,Aug 2025,FT,Nice,Toulouse,613,246,0,1,2025-08-16,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
4,74,22,10792,24609,France Ligue 1,2025-2026,1911281,https://www.whoscored.com/matches/1911281/live...,Aug 2025,FT,Brest,Lille,2332,607,3,3,2025-08-17,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,74,22,10792,24609,France Ligue 1,2025-2026,1911524,https://www.whoscored.com/matches/1911524/live...,May 2026,FT,Marseille,Rennes,249,313,3,1,2026-05-17,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,34,34
302,74,22,10792,24609,France Ligue 1,2025-2026,1911525,https://www.whoscored.com/matches/1911525/live...,May 2026,FT,Nantes,Toulouse,302,246,0,0,2026-05-17,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,34,34
303,74,22,10792,24609,France Ligue 1,2025-2026,1911526,https://www.whoscored.com/matches/1911526/live...,May 2026,FT,Nice,Metz,613,314,0,0,2026-05-17,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,34,34
304,74,22,10792,24609,France Ligue 1,2025-2026,1911527,https://www.whoscored.com/matches/1911527/live...,May 2026,FT,Paris FC,Paris Saint-Germain,2832,304,2,1,2026-05-17,https://d2zywfiolv4f83.cloudfront.net/img/team...,https://d2zywfiolv4f83.cloudfront.net/img/team...,34,34


In [202]:
df_ligue1['matchweek_id'].value_counts().sort_index()

matchweek_id
1     9
2     9
3     9
4     9
5     9
6     9
7     9
8     9
9     9
10    9
11    9
12    9
13    9
14    9
15    9
16    9
17    9
18    9
19    9
20    9
21    9
22    9
23    9
24    9
25    9
26    9
27    9
28    9
29    9
30    9
31    9
32    9
33    9
34    9
Name: count, dtype: Int64

In [204]:
cols = ["homeTeamName", "awayTeamName"]
df_ligue1[cols] = df_ligue1[cols].replace(MAPPING_NAMES_WHOSCORED_TO_FOTMOB)

In [207]:
compare_team_names(ligue1, "name_team_fotmob",df_ligue1, "homeTeamName")

Total únicos name_team_fotmob: 18
Total únicos homeTeamName: 18

Solo en name_team_fotmob:
None

Solo en homeTeamName:
None

Coinciden: 18


{'only_in_col1': set(),
 'only_in_col2': set(),
 'intersection': {'Angers',
  'Auxerre',
  'Brest',
  'Le Havre',
  'Lens',
  'Lille',
  'Lorient',
  'Lyon',
  'Marseille',
  'Metz',
  'Monaco',
  'Nantes',
  'Nice',
  'Paris FC',
  'Paris Saint-Germain',
  'Rennes',
  'Strasbourg',
  'Toulouse'}}

In [208]:
compare_team_names(ligue1, "name_team_fotmob",df_ligue1, "awayTeamName")

Total únicos name_team_fotmob: 18
Total únicos awayTeamName: 18

Solo en name_team_fotmob:
None

Solo en awayTeamName:
None

Coinciden: 18


{'only_in_col1': set(),
 'only_in_col2': set(),
 'intersection': {'Angers',
  'Auxerre',
  'Brest',
  'Le Havre',
  'Lens',
  'Lille',
  'Lorient',
  'Lyon',
  'Marseille',
  'Metz',
  'Monaco',
  'Nantes',
  'Nice',
  'Paris FC',
  'Paris Saint-Germain',
  'Rennes',
  'Strasbourg',
  'Toulouse'}}

In [ ]:
cols = ["homeTeamName", "awayTeamName"]

df_ligue1[cols] = df_ligue1[cols].replace(MAPPING_NAMES_WHOSCORED_TO_FOTMOB)

In [ ]:
compare_team_names(ligue1, "name_team_fotmob",df_ligue1, "homeTeamName")

Total únicos name_team_fotmob: 18
Total únicos homeTeamName: 18

Solo en name_team_fotmob:
None

Solo en homeTeamName:
None

Coinciden: 18


{'only_in_col1': set(),
 'only_in_col2': set(),
 'intersection': {'Angers',
  'Auxerre',
  'Brest',
  'Le Havre',
  'Lens',
  'Lille',
  'Lorient',
  'Lyon',
  'Marseille',
  'Metz',
  'Monaco',
  'Nantes',
  'Nice',
  'Paris FC',
  'Paris Saint-Germain',
  'Rennes',
  'Strasbourg',
  'Toulouse'}}

In [ ]:
compare_team_names(ligue1, "name_team_fotmob",df_ligue1, "awayTeamName")

Total únicos name_team_fotmob: 18
Total únicos awayTeamName: 18

Solo en name_team_fotmob:
None

Solo en awayTeamName:
None

Coinciden: 18


{'only_in_col1': set(),
 'only_in_col2': set(),
 'intersection': {'Angers',
  'Auxerre',
  'Brest',
  'Le Havre',
  'Lens',
  'Lille',
  'Lorient',
  'Lyon',
  'Marseille',
  'Metz',
  'Monaco',
  'Nantes',
  'Nice',
  'Paris FC',
  'Paris Saint-Germain',
  'Rennes',
  'Strasbourg',
  'Toulouse'}}

PLAYERS FOTMOB

In [25]:
def build_team_logo(ccode):

    if pd.isna(ccode):
        return None

    ccode = str(ccode).strip()

    # Si son letras (ESP, CZE, ARG...)
    if ccode.isalpha():
        return (f"https://images.fotmob.com/image_resources/logo/teamlogo/{ccode.lower()}.png")

    # Si es un id numérico de club
    return (f"https://images.fotmob.com/image_resources/logo/teamlogo/{ccode}.png")

In [26]:
def extract_all_players_wc26_fotmob(df_all_teams_league_venues_photo):
    # Obtener dataframe de equipos
    all_players = []

    for _, row in tqdm(df_all_teams_league_venues_photo.iterrows(), total=len(df_all_teams_league_venues_photo), desc="Descargando plantillas"):

        team_id = row["id_team_fotmob"]

        # Cambia "name" por el nombre real de la columna de equipo si es distinto
        team_name = row.get("name_team_fotmob", None)

        try:
            response = requests.get( f"https://www.fotmob.com/api/data/teams?id={team_id}&ccode3=ESP", timeout=10)
            response.raise_for_status()

            data = response.json()

            # Saltar equipos sin plantilla
            if (
                "squad" not in data
                or data["squad"] is None
                or "squad" not in data["squad"]
            ):
                continue

            # Secciones de la plantilla (Porteros, Defensas, etc.)
            squad_df = pd.json_normalize(data["squad"]["squad"])
            if squad_df.empty:
                continue

            squad_df = squad_df.explode("members").reset_index(drop=True)

            # Información de jugadores
            players_df = pd.json_normalize(squad_df["members"])

            # Unir posición/grupo con datos del jugador
            team_players = pd.concat( [squad_df[["title"]], players_df], axis=1)

            # Añadir información del equipo
            team_players["team_id"] = team_id
            team_players["team_name"] = team_name

            all_players.append(team_players)

        except Exception as e:
            print(f"❌ Error en equipo {team_id}: {e}")

    # Unir todos los jugadores
    final_df = pd.concat(all_players, ignore_index=True)

    # Eliminar columnas innecesarias si existen
    final_df = final_df.drop( columns=[ "title", "role.key", "injury","excludeFromRanking"], errors="ignore")
    final_df = final_df.dropna(subset=["id"])
    # URL de la foto del jugador
    final_df["member_photo"] = "https://images.fotmob.com/image_resources/playerimages/" + final_df["id"].astype(int).astype(str)+ ".png"
    final_df["team_logo"] = final_df["ccode"].apply(build_team_logo)
    final_df["id"] = final_df["id"].astype(float).astype(int).astype(str)

    # Opcional: reordenar columnas principales
    cols_first = [ "team_id", "team_name","id","name","positionId","shirtNumber","member_photo"]

    existing_cols = [c for c in cols_first if c in final_df.columns]
    remaining_cols = [c for c in final_df.columns if c not in existing_cols]

    final_df = final_df[existing_cols + remaining_cols]

    print(f"✅ Total jugadores: {len(final_df):,}")
    print(f"✅ Total equipos: {final_df['team_id'].nunique():,}")
    return final_df

In [27]:
all_players_5leagues=extract_all_players_wc26_fotmob(df_all_teams_league_venues_photo)
all_players_5leagues

Descargando plantillas: 100%|██████████| 96/96 [00:35<00:00,  2.73it/s]

✅ Total jugadores: 3,223
✅ Total equipos: 96


,team_id,team_name,id,name,positionId,shirtNumber,member_photo,height,age,dateOfBirth,...,assists,rcards,ycards,positionIds,positionIdsDesc,transferValue,injured,injury.id,injury.expectedReturn,team_logo
0,9825,Arsenal,24011,Mikel Arteta,NaN,NaN,https://images.fotmob.com/image_resources/play...,176.0,44.0,1982-03-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://images.fotmob.com/image_resources/logo...
1,9825,Arsenal,562727,David Raya,0.0,1.0,https://images.fotmob.com/image_resources/play...,183.0,30.0,1995-09-15,...,0.0,0.0,0.0,11,GK,27458190.0,NaN,NaN,NaN,https://images.fotmob.com/image_resources/logo...
2,9825,Arsenal,317564,Kepa Arrizabalaga,0.0,13.0,https://images.fotmob.com/image_resources/play...,189.0,31.0,1994-10-03,...,0.0,0.0,0.0,11,GK,5423054.0,NaN,NaN,NaN,https://images.fotmob.com/image_resources/logo...
3,9825,Arsenal,1243239,Tommy Setford,0.0,35.0,https://images.fotmob.com/image_resources/play...,185.0,20.0,2006-03-13,...,0.0,0.0,0.0,11,GK,NaN,NaN,NaN,NaN,https://images.fotmob.com/image_resources/logo...
4,9825,Arsenal,1137667,Piero Hincapié,1.0,NaN,https://images.fotmob.com/image_resources/play...,184.0,24.0,2002-01-09,...,0.0,0.0,0.0,"38,36","LB,CB",56767496.0,NaN,NaN,NaN,https://images.fotmob.com/image_resources/logo...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3227,8670,Real Oviedo,1370087,Pablo Sáenz,3.0,NaN,https://images.fotmob.com/image_resources/play...,175.0,25.0,2001-05-22,...,0.0,0.0,0.0,"103,104","RW,ST",433342.0,NaN,NaN,NaN,https://images.fotmob.com/image_resources/logo...
3228,8670,Real Oviedo,1169595,Ilyas Chaira,3.0,7.0,https://images.fotmob.com/image_resources/play...,179.0,25.0,2001-02-02,...,0.0,0.0,0.0,"87,78,83,72","LW,LM,RW,RM",3699372.0,NaN,NaN,NaN,https://images.fotmob.com/image_resources/logo...
3229,8670,Real Oviedo,971787,Haissem Hassan,3.0,10.0,https://images.fotmob.com/image_resources/play...,173.0,24.0,2002-02-08,...,0.0,0.0,0.0,"83,72","RW,RM",3400496.0,NaN,NaN,NaN,https://images.fotmob.com/image_resources/logo...
3230,8670,Real Oviedo,859764,Luka Ilic,3.0,21.0,https://images.fotmob.com/image_resources/play...,182.0,27.0,1999-07-02,...,0.0,0.0,0.0,104,ST,1299010.0,NaN,NaN,NaN,https://images.fotmob.com/image_resources/logo...


RANKING FIFA

In [23]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd

URL = "https://www.uefa.com/nationalassociations/uefarankings/club/?year=2027"

driver = webdriver.Chrome()
driver.maximize_window()
driver.get(URL)
time.sleep(6)

# --------------------------------------------------
# 1. COOKIES
# --------------------------------------------------
try:
    wait = WebDriverWait(driver, 10)
    cookies_btn = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "a.cmpboxbtnyes")
    ))
    driver.execute_script("arguments[0].click();", cookies_btn)
    print("✓ Cookies aceptadas")
    time.sleep(3)
except Exception as e:
    print(f"✗ Cookies: {e}")

# --------------------------------------------------
# 2. CLICK EN "VIEW FULL RANKINGS"
# --------------------------------------------------
try:
    buttons = driver.find_elements(By.TAG_NAME, "pk-button")
    btn = next((b for b in buttons if "View full rankings" in b.text), None)
    if btn:
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
        time.sleep(1)
        ActionChains(driver).move_to_element(btn).pause(0.3).click().perform()
        print("✓ Botón clicado")
        time.sleep(4)
    else:
        print("✗ Botón no encontrado")
except Exception as e:
    print(f"✗ Error botón: {e}")

# --------------------------------------------------
# 3. EXTRACCIÓN VÍA BEAUTIFULSOUP
# --------------------------------------------------
def extract_from_html(html, processed_ids, rows_data):
    soup = BeautifulSoup(html, "html.parser")
    rows = soup.select("div[row-id]")
    nuevas = 0
    for row in rows:
        row_id = row.get("row-id")
        if not row_id or row_id in processed_ids:
            continue
        data = {}
        pos_cell = row.select_one('[col-id="position"] .ag-cell-value')
        if pos_cell:
            data["position"] = pos_cell.get_text(strip=True)
        icon = row.select_one('[col-id="position"] pk-icon')
        if icon:
            name = icon.get("name", "")
            data["movement"] = "UP" if "in" in name else "DOWN" if "out" in name else "SAME"
        else:
            data["movement"] = "SAME"
        club = row.select_one('[col-id="member"] [slot="primary"] span')
        if club:
            data["club"] = club.get_text(strip=True)
        country = row.select_one('[col-id="member"] [slot="secondary"]')
        if country:
            data["country"] = country.get_text(strip=True)
        for col in ["season-2023", "season-2024", "season-2025",
                    "season-2026", "season-2027", "points", "na"]:
            cell = row.select_one(f'[col-id="{col}"] .ag-cell-value')
            if cell:
                data[col] = cell.get_text(strip=True)
        rows_data[row_id] = data
        processed_ids.add(row_id)
        nuevas += 1
    return nuevas

# --------------------------------------------------
# 4. SCROLL ADAPTATIVO — espera solo lo necesario
# --------------------------------------------------
SCROLL_STEP = 600
MAX_STALLS  = 3
MAX_WAIT    = 1.0    # timeout máximo esperando nuevas filas
POLL        = 0.12   # intervalo de comprobación

def get_scroll_height():
    return driver.execute_script("return document.documentElement.scrollHeight;")

def wait_for_dom_change(initial_sh, timeout=MAX_WAIT):
    """Retorna en cuanto scrollHeight cambia, o cuando pasa el timeout."""
    start = time.time()
    while time.time() - start < timeout:
        if get_scroll_height() != initial_sh:
            return True   # DOM cambió
        time.sleep(POLL)
    return False  # timeout

rows_data     = {}
processed_ids = set()

driver.execute_script("window.scrollTo(0, 0);")
time.sleep(0.5)

stalls      = 0
last_scroll = -1
iteration   = 0
t0          = time.time()

last_count  = 0
no_new_streak = 0  # iteraciones consecutivas sin nuevos registros

while True:
    iteration += 1

    grid_html = driver.execute_script(
        "return document.querySelector('.ag-root-wrapper')?.innerHTML || '';"
    )
    nuevas = extract_from_html(grid_html, processed_ids, rows_data)

    top = driver.execute_script("return window.pageYOffset;")
    sh  = get_scroll_height()
    ch  = driver.execute_script("return window.innerHeight;")

    print(f"[{iteration:03d}] {time.time()-t0:5.1f}s | scrollTop={int(top)} | "
          f"scrollH={sh} | registros={len(rows_data)} | nuevas={nuevas}")

    # Condición 1: fondo real
    if top + ch >= sh - 5:
        print("✓ Fondo alcanzado")
        break

    # Condición 2: sin nuevos registros N veces seguidas → ya terminamos
    if nuevas == 0:
        no_new_streak += 1
        if no_new_streak >= 3:
            print(f"✓ Sin nuevos registros en {no_new_streak} iteraciones, terminando")
            break
    else:
        no_new_streak = 0

    # Stall de scroll (posición no avanza)
    if int(top) == int(last_scroll):
        stalls += 1
        if stalls >= MAX_STALLS:
            print("✗ Scroll bloqueado, parando")
            break
        time.sleep(0.3)
    else:
        stalls = 0

    last_scroll = int(top)
    sh_before = get_scroll_height()
    driver.execute_script(f"window.scrollBy(0, {SCROLL_STEP});")

    changed = wait_for_dom_change(sh_before, timeout=1.0)  # MAX_WAIT=1.0
    if not changed:
        time.sleep(0.1)

# Pasada final
grid_html = driver.execute_script(
    "return document.querySelector('.ag-root-wrapper')?.innerHTML || '';"
)
extract_from_html(grid_html, processed_ids, rows_data)

# --------------------------------------------------
# 5. RESULTADO
# --------------------------------------------------
df_all_ranking_fifa_teams = pd.DataFrame(rows_data.values())
print(f"\nTotal registros extraídos: {len(df_all_ranking_fifa_teams)}")
print(f"Tiempo total: {time.time()-t0:.1f}s")
driver.quit()


✓ Cookies aceptadas
✓ Botón clicado
[001]   0.4s | scrollTop=0 | scrollH=2982 | registros=20 | nuevas=20
[002]   1.7s | scrollTop=600 | scrollH=2982 | registros=20 | nuevas=0
[003]   2.3s | scrollTop=1200 | scrollH=3622 | registros=30 | nuevas=10
[004]   2.9s | scrollTop=1800 | scrollH=4262 | registros=40 | nuevas=10
[005]   3.7s | scrollTop=2400 | scrollH=4902 | registros=50 | nuevas=10
[006]   4.4s | scrollTop=3000 | scrollH=5542 | registros=60 | nuevas=10
[007]   5.1s | scrollTop=3600 | scrollH=6182 | registros=70 | nuevas=10
[008]   6.0s | scrollTop=4200 | scrollH=6822 | registros=80 | nuevas=10
[009]   6.8s | scrollTop=4800 | scrollH=7462 | registros=90 | nuevas=10
[010]   7.8s | scrollTop=5400 | scrollH=8102 | registros=100 | nuevas=10
[011]   8.7s | scrollTop=6000 | scrollH=8742 | registros=110 | nuevas=10
[012]   9.8s | scrollTop=6600 | scrollH=9382 | registros=120 | nuevas=10
[013]  11.3s | scrollTop=7200 | scrollH=10022 | registros=130 | nuevas=10
[014]  12.6s | scrollTop=780

In [24]:
df_all_ranking_fifa_teams

,position,movement,club,country,season-2023,season-2024,season-2025,season-2026,season-2027,points,na
0,1,UP,Bayern München,Germany,27.000,28.000,27.250,39.250,-,121.500,15.337
1,2,UP,Arsenal,England,17.000,22.000,36.000,44.000,-,119.000,19.703
2,3,DOWN,Real Madrid,Spain,29.000,34.000,24.500,27.000,-,114.500,15.723
3,4,UP,Paris,France,19.000,23.000,33.500,37.500,-,113.000,13.016
4,5,DOWN,Inter,Italy,29.000,20.000,40.250,19.750,-,109.000,16.846
...,...,...,...,...,...,...,...,...,...,...,...
379,380,SAME,Glentoran,Northern Ireland,-,1.000,-,-,-,1.125,1.125
380,381,SAME,Fola,Luxembourg,1.000,-,-,-,-,1.075,1.075
381,382,SAME,Kalev,Estonia,-,-,1.000,-,-,1.000,0.908
382,383,SAME,Trans,Estonia,-,1.000,-,-,-,1.000,0.908


Last updated: 03/06/2026 09:12

In [ ]:
countries = ["Germany", "England", "Spain", "France", "Italy"]

df_ranking_fifa_only5biglueagues = df_all_ranking_fifa_teams[df_all_ranking_fifa_teams["country"].isin(countries)].reset_index(drop= True)
df_ranking_fifa_only5biglueagues

,position,movement,club,country,season-2023,season-2024,season-2025,season-2026,season-2027,points,na
0,1,UP,Bayern München,Germany,27.000,28.000,27.250,39.250,-,121.500,15.337
1,2,UP,Arsenal,England,17.000,22.000,36.000,44.000,-,119.000,19.703
2,3,DOWN,Real Madrid,Spain,29.000,34.000,24.500,27.000,-,114.500,15.723
3,4,UP,Paris,France,19.000,23.000,33.500,37.500,-,113.000,13.016
4,5,DOWN,Inter,Italy,29.000,20.000,40.250,19.750,-,109.000,16.846
5,6,DOWN,Man City,England,33.000,28.000,14.750,22.750,-,98.500,19.703
6,7,UP,Barcelona,Spain,9.000,23.000,36.250,30.000,-,98.250,15.723
7,8,DOWN,Liverpool,England,19.000,20.000,29.500,28.500,-,97.000,19.703
8,9,UP,Leverkusen,Germany,19.000,29.000,23.250,19.750,-,91.000,15.337
9,10,DOWN,B. Dortmund,Germany,18.000,29.000,27.750,16.000,-,90.750,15.337


In [ ]:
compare_team_names(df_all_teams_league_venues_photo, "name_team_fotmob",df_ranking_fifa_only5biglueagues, "club")

Total únicos name_team_fotmob: 96
Total únicos club: 58

Solo en name_team_fotmob:
{'Celta Vigo', 'Werder Bremen', 'St. Pauli', 'Manchester City', 'Le Havre', 'Cremonese', 'Borussia Dortmund', 'Leeds United', 'Sunderland', 'Getafe', 'Paris Saint-Germain', 'VfB Stuttgart', 'Elche', 'Eintracht Frankfurt', 'Burnley', '1. FC Köln', 'Brighton & Hove Albion', 'Atletico Madrid', 'West Ham United', 'Mainz 05', 'Sassuolo', 'Brentford', 'Hellas Verona', 'Real Oviedo', 'Manchester United', 'Espanyol', 'RB Leipzig', 'Auxerre', 'Genoa', 'Torino', 'Bayer Leverkusen', 'Fulham', 'Como', 'Mallorca', 'Lorient', 'FC Heidenheim', 'Everton', 'Lecce', 'Newcastle United', 'Paris FC', 'Udinese', 'Levante', 'Deportivo Alaves', 'Nottingham Forest', 'Tottenham Hotspur', 'Metz', 'Hamburger SV', 'Valencia', 'Wolverhampton Wanderers', 'Borussia Mönchengladbach', 'Wolfsburg', 'Augsburg', 'Pisa', 'Angers', 'Cagliari', 'Parma', 'AFC Bournemouth'}

Solo en club:
{'Heidenheim', 'Man Utd', 'Stuttgart', 'Celta', 'Köln', '

{'only_in_col1': {'1. FC Köln',
  'AFC Bournemouth',
  'Angers',
  'Atletico Madrid',
  'Augsburg',
  'Auxerre',
  'Bayer Leverkusen',
  'Borussia Dortmund',
  'Borussia Mönchengladbach',
  'Brentford',
  'Brighton & Hove Albion',
  'Burnley',
  'Cagliari',
  'Celta Vigo',
  'Como',
  'Cremonese',
  'Deportivo Alaves',
  'Eintracht Frankfurt',
  'Elche',
  'Espanyol',
  'Everton',
  'FC Heidenheim',
  'Fulham',
  'Genoa',
  'Getafe',
  'Hamburger SV',
  'Hellas Verona',
  'Le Havre',
  'Lecce',
  'Leeds United',
  'Levante',
  'Lorient',
  'Mainz 05',
  'Mallorca',
  'Manchester City',
  'Manchester United',
  'Metz',
  'Newcastle United',
  'Nottingham Forest',
  'Paris FC',
  'Paris Saint-Germain',
  'Parma',
  'Pisa',
  'RB Leipzig',
  'Real Oviedo',
  'Sassuolo',
  'St. Pauli',
  'Sunderland',
  'Torino',
  'Tottenham Hotspur',
  'Udinese',
  'Valencia',
  'VfB Stuttgart',
  'Werder Bremen',
  'West Ham United',
  'Wolfsburg',
  'Wolverhampton Wanderers'},
 'only_in_col2': {'Atleti

How men's club coefficients are calculated


The text that follows has been taken from Annex D of the official UEFA Champions League regulations, and appears in the same place in the official UEFA Europa League regulations and the official UEFA Conference League regulations.

D.1 System overview
UEFA calculates the coefficient of each club each season based on the clubs' results in the UEFA Champions League, UEFA Europa League and UEFA Conference League. The season coefficients from the five most recent seasons are used to rank the clubs for seeding purposes (sporting club coefficient). In addition, the season coefficients from the ten most recent seasons are used to calculate revenue club coefficients for revenue distribution purposes only.

D.2 Reference periods for rankings
The five-season sporting club coefficients for the 2026/27 UEFA Champions League, UEFA Europa League and UEFA Conference League are established before the start of the 2026/27 season, on the basis of each club's season coefficients from seasons 2021/22 to 2025/26 inclusive.

The ten-season revenue club coefficients for the 2026/27 UEFA Champions League, UEFA Europa League and UEFA Conference League are established before the start of the 2026/27 season, on the basis of each club's season coefficients from seasons 2016/17 to 2025/26 inclusive.

D.4 Sporting club coefficient calculation
The season coefficient of a club is calculated by adding up the total number of points it obtained in a given season of the UEFA Champions League, UEFA Europa League or UEFA Conference League in accordance with this Annex D of the competition regulations for that season. A club’s five-season sporting coefficient is the cumulative total of its five season coefficients from the reference period stipulated in Annex D.2, or 20% of its association’s five-season association coefficient, whichever is higher. Points awarded each season are in accordance with the relevant competition regulations for that specific season, so Annex D.4.1 to Annex D.4.3 apply only to the awarding of points in the 2023/24 season.

D.4.1 Points awarded in the UEFA Champions League

a. Qualifying phase and play-offs:

o Clubs that are eliminated in the UEFA Champions League qualifying phase or play-offs are awarded points in the UEFA Europa League or UEFA Conference League (see Annex D.4.2 and Annex D.4.3).

b. League phase onwards (except knockout phase play-offs):

o 2 points for a win;

o 1 point for a draw;

o 0 points for a defeat.

D.4.2 Points awarded in the UEFA Europa League
a. Qualifying phase and play-offs:

o Clubs that are eliminated in the UEFA Europa League qualifying phase or play-offs are awarded points in the UEFA Conference League (see Annex D.4.3).

b. League phase onwards (except knockout phase play-offs):

o 2 points for a win;

o 1 point for a draw;

o 0 points for a defeat.

c. Guaranteed minimum:

Clubs are guaranteed a minimum of three points in the league phase of the UEFA Europa League even if the number of points actually obtained during this stage is lower. This guaranteed minimum is not added to points actually obtained in the league phase and is not included in the association coefficient calculation.

D.4.3 Points awarded in the UEFA Conference League
a. Qualifying phase and play-offs:

o 1 point awarded to each club eliminated in the first qualifying round;

o 1.5 points awarded to each club eliminated in the second qualifying round;

o 2 points awarded to each club eliminated in the third qualifying round;

o 2.5 points awarded to each club eliminated in the play-offs.

b. League phase onwards (except knockout phase play-offs):

o 2 points for a win;

o 1 point for a draw;

o 0 points for a defeat.

C. Guaranteed minimum:

Clubs are guaranteed a minimum of 2.5 points in the league phase of the UEFA Conference League, even if the number of points actually obtained during this stage is lower. This guaranteed minimum is not added to points actually obtained in the league phase and is not included in the association coefficient calculation.

D.5 Bonus points
Clubs are awarded the following bonus points based on their final position in the league phase rankings:


Clubs that reach the round of 16, quarter-finals, semi-finals or final are awarded 1.5 extra points for each such round in the UEFA Champions League, 1 extra point for each such round in the UEFA Europa League, and 0.5 extra points for each such round in the UEFA Conference League.

D.6 Revenue club coefficient calculation
A club’s revenue club coefficient is calculated using the same season coefficients as for the sporting club coefficient (see Annex D.4), but with a ten-season instead of a five-season reference period as stipulated in Annex D.2.

A club’s ten-season revenue coefficient is the cumulative total of its ten season coefficients from the reference period stipulated in Annex D.2, or 20% of the equivalent ten-season total of its association's season coefficients, whichever is higher.

D.7 Calculation principles
Match points are awarded based on the final scores ratified by UEFA. Penalty shoot-outs do not count.

Coefficients are calculated to the thousandth and not rounded up.

If competition rounds are changed and matches are played as single-leg instead of two-legged ties, the points awarded under Annex D.3, Annex D.4.1.b, Annex D.4.2.b and Annex D.4.3.b are adapted as follows:

a. 3 points for a win in regular time or extra time (1.5 points for qualifying and play-off matches);

b. 2 points for a draw in extra time (1 point for qualifying and play-off matches);

c. 1 point for a defeat in regular time or extra time (0.5 points for qualifying and play-off matches).

This does not apply to the final of the UEFA Champions League, UEFA Europa League or UEFA Conference League.

D.8 Equal coefficients
If two or more clubs are ranked equally, the following criteria are applied, in this order, to determine their final rankings:

• their coefficients in the most recent of the seasons on which the rankings are based;

• their coefficients in the next most recent season in which they are not equal;

• the association coefficient of the club’s respective association;

• their position in the domestic championship in the most recent season.

D.9 Final decisions
The UEFA administration takes final decisions on any matters not provided for in this annex.

ELO BESOCCER

In [20]:
def get_elo_from_team_page(url: str, headers: dict) -> int | None:
    try:
        r = requests.get(url, headers=headers)
        soup = BeautifulSoup(r.text, "html.parser")
        elo_span = soup.select_one("div.elo-box div.elo span")
        if elo_span:
            val = elo_span.get_text(strip=True)
            return int(val) if val.isdigit() else None
    except Exception:
        pass
    return None


def scrape_besoccer_teams(url: str) -> pd.DataFrame:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
        "Referer": "https://www.google.com/",
    }

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    season_match = re.search(r"/(\d{4})$", url)
    league_match = re.search(r"/info/([^/]+)/", url)
    season = int(season_match.group(1)) if season_match else None
    league = league_match.group(1) if league_match else None

    if league =='primera_division':
        league= 'LaLiga'
    elif league == 'bundesliga':
        league= "Bundesliga"
    elif league=='ligue_1':
        league = 'Ligue 1'
    elif league == 'serie_a':
        league = 'Serie a'
    elif league== "premier_league":
        league = "Premier League"

    panel = soup.select_one("#mod_teams")
    if not panel:
        return pd.DataFrame(columns=["position", "team_name_besoccer", "team_id_besoccer",
                                     "elo_besoccer", "url_team_besoccer", "league", "season"])

    equipos = []
    for a in panel.select("a[data-cy='compTeam']"):
        nombre   = a.select_one(".name span")
        position = a.select_one(".team-pl")
        elo      = a.select_one(".rating-box strong")
        img      = a.select_one("img")

        escudo_url    = img.get("src") if img else None
        team_id_match = re.search(r"/(\d+)\.jpg", escudo_url) if escudo_url else None
        elo_val       = elo.get_text(strip=True) if elo else ""

        equipos.append({
            "position":           position.get_text(strip=True).replace("º", "") if position else None,
            "team_name_besoccer": nombre.get_text(strip=True) if nombre else None,
            "team_id_besoccer":   team_id_match.group(1) if team_id_match else None,
            "elo_besoccer":       int(elo_val) if elo_val.isdigit() else None,
            "url_team_besoccer":  a.get("href"),
            "league":             league,
            "season":             season,
        })

    df = pd.DataFrame(equipos)
    df["position"]    = pd.to_numeric(df["position"],    errors="coerce").astype("Int64")
    df["elo_besoccer"] = pd.to_numeric(df["elo_besoccer"], errors="coerce").astype("Int64")

    # Fallback: si todos los ELO son NaN, sacarlos de cada página de equipo
    if df["elo_besoccer"].isna().all():
        print(f"ELO no disponible en panel para {league}, scrapeando páginas de equipo...")
        elos = []
        for team_url in df["url_team_besoccer"]:
            elo_val = get_elo_from_team_page(team_url, headers)
            elos.append(elo_val)
            time.sleep(0.5)  # cortesía para no banear
        df["elo_besoccer"] = pd.array(elos, dtype="Int64")

    return df

MAPPING_TEAM_NAME_BESOCCER_TO_FOTMOB= {

    'Heidenheim': 'FC Heidenheim', 
    'Atlético de Madrid': 'Atletico Madrid', 
    'FC Barcelona': 'Barcelona', 
    'Stuttgart': 'VfB Stuttgart', 
    'Athletic': 'Athletic Club', 
    'Celta': 'Celta Vigo', 
    'Olympique Marseille': 'Marseille', 
    'Köln': '1. FC Köln', 
    'SC Freiburg': 'Freiburg', 
    'FC St Pauli': 'St. Pauli', 
    'FC Augsburg': 'Augsburg', 
    'Angers SCO': 'Angers', 
    'Wolves': 'Wolverhampton Wanderers', 
    'Stade Rennais': 'Rennes', 
    'Olympique Lyonnais': 'Lyon', 
    'PSG': 'Paris Saint-Germain', 
    'Girona FC': 'Girona', 
    'West Ham': 'West Ham United', 
    'Stade Brestois': 'Brest', 
    'B. Leverkusen': 'Bayer Leverkusen', 
    'Pisa SC': 'Pisa', 
    'B. Mönchengladbach': 'Borussia Mönchengladbach', 
    'Deportivo Alavés': 'Deportivo Alaves', 
    'B. Dortmund': 'Borussia Dortmund', 
    'Newcastle': 'Newcastle United'
}



def scrape_multiple_leagues_elo(year_last) -> pd.DataFrame:
    urls = [
        f"https://www.besoccer.com/competition/info/bundesliga/{year_last}",
        f"https://www.besoccer.com/competition/info/primera_division/{year_last}",
        f"https://www.besoccer.com/competition/info/ligue_1/{year_last}",
        f"https://www.besoccer.com/competition/info/serie_a/{year_last}",
        f"https://www.besoccer.com/competition/info/premier_league/{year_last}",
    ]

    dfs = [scrape_besoccer_teams(url) for url in urls]
    df_all_teams_elo_besoccer = pd.concat(dfs, ignore_index=True)
    df_all_teams_elo_besoccer['team_name_besoccer'] = df_all_teams_elo_besoccer['team_name_besoccer'].replace(MAPPING_TEAM_NAME_BESOCCER_TO_FOTMOB)
    return df_all_teams_elo_besoccer

In [21]:
df_all_teams_elo_besoccer = scrape_multiple_leagues_elo("2026")
df_all_teams_elo_besoccer

ELO no disponible en panel para Bundesliga, scrapeando páginas de equipo...
ELO no disponible en panel para LaLiga, scrapeando páginas de equipo...
ELO no disponible en panel para Ligue 1, scrapeando páginas de equipo...
ELO no disponible en panel para Serie a, scrapeando páginas de equipo...
ELO no disponible en panel para Premier League, scrapeando páginas de equipo...


,position,team_name_besoccer,team_id_besoccer,elo_besoccer,url_team_besoccer,league,season
0,1,Bayern München,449,94,https://www.besoccer.com/team/bayern-munchen,Bundesliga,2026
1,2,Borussia Dortmund,530,90,https://www.besoccer.com/team/borussia-dortmund,Bundesliga,2026
2,3,RB Leipzig,9750,89,https://www.besoccer.com/team/rb-leipzig,Bundesliga,2026
3,4,VfB Stuttgart,2433,88,https://www.besoccer.com/team/stuttgart,Bundesliga,2026
4,5,Hoffenheim,2542,84,https://www.besoccer.com/team/tsg-1899-hoffenheim,Bundesliga,2026
...,...,...,...,...,...,...,...
91,16,Nottingham Forest,1825,85,https://www.besoccer.com/team/nottingham-fores...,Premier League,2026
92,17,Tottenham Hotspur,2523,88,https://www.besoccer.com/team/tottenham-hotspu...,Premier League,2026
93,18,West Ham United,2804,84,https://www.besoccer.com/team/west-ham-united,Premier League,2026
94,19,Burnley,569,84,https://www.besoccer.com/team/burnley-fc,Premier League,2026


Last updated: 29/06/2026 12:12

TRANSFERENCIAS POR EQUIPOS

In [271]:
import time
import pandas as pd
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup


NAME_LEAGUE_TRANSFERMARKT = {
    "LaLiga": {
        "code": "ES1",
        "slug": "laliga"
    },
    "Bundesliga": {
        "code": "L1",
        "slug": "bundesliga"
    },
    "Ligue 1": {
        "code": "FR1",
        "slug": "ligue-1"
    },
    "Serie A": {
        "code": "IT1",
        "slug": "serie-a"
    },
    "Premier League": {
        "code": "GB1",
        "slug": "premier-league"
    }
}

MAPPING_TEAM_NAME_TRANSFERMARKET_TO_FOTMOB= {

    '1.FC Köln': '1. FC Köln', 
    'Genoa CFC': 'Genoa', 
    'AS Roma':'Roma', 
    'Levante UD': 'Levante', 
    'Bayer 04 Leverkusen': 'Bayer Leverkusen', 
    'Elche CF': 'Elche', 
    'Celta de Vigo': 'Celta Vigo', 
    'Girona FC': 'Girona', 
    'Torino FC': 'Torino', 
    'Valencia CF': 'Valencia', 
    'Cagliari Calcio': 'Cagliari', 
    'Olympique Lyon': 'Lyon', 
    '1.FC Heidenheim 1846': 'FC Heidenheim', 
    'Bologna FC 1909': 'Bologna', 
    'ACF Fiorentina': 'Fiorentina', 
    'US Lecce': 'Lecce', 
    'Burnley FC': 'Burnley', 
    'FC St. Pauli':'St. Pauli', 
    'AS Monaco': 'Monaco', 
    'FC Toulouse':'Toulouse', 
    'Chelsea FC': 'Chelsea', 
    'Bayern Munich': 'Bayern München', 
    'Atlético de Madrid': 'Atletico Madrid', 
    'Udinese Calcio': 'Udinese', 
    'TSG 1899 Hoffenheim': 'Hoffenheim', 
    'Villarreal CF': 'Villarreal', 
    'VfL Wolfsburg': 'Wolfsburg', 
    'FC Barcelona': 'Barcelona', 
    'FC Augsburg': 'Augsburg', 
    'FC Lorient': 'Lorient', 
    'SS Lazio': 'Lazio', 
    'Juventus FC':'Juventus', 
    'FC Metz': 'Metz', 
    'Getafe CF': 'Getafe', 
    'SV Werder Bremen': 'Werder Bremen', 
    'LOSC Lille':  'Lille', 
    'Olympique Marseille': 'Marseille', 
    'Liverpool FC':'Liverpool', 
    'RCD Espanyol Barcelona': 'Espanyol', 
    'Brentford FC': 'Brentford',
    'RC Strasbourg Alsace': 'Strasbourg', 
    'OGC Nice': 'Nice', 
    'Athletic Bilbao': 'Athletic Club', 
    'Pisa Sporting Club': 'Pisa', 
    'FC Nantes': 'Nantes', 
    'Sevilla FC': 'Sevilla', 
    'RC Lens': 'Lens', 
    'Fulham FC': 'Fulham', 
    'Stade Rennais FC': 'Rennes', 
    'Le Havre AC': 'Le Havre', 
    'Deportivo Alavés': 'Deportivo Alaves', 
    '1.FC Union Berlin': 'Union Berlin', 
    'SSC Napoli': 'Napoli', 
    'Real Betis Balompié': 'Real Betis', 
    'Inter Milan':'Inter', 
    'Como 1907': 'Como', 
    'Arsenal FC': 'Arsenal', 
    'Everton FC':'Everton', 
    'RCD Mallorca':'Mallorca', 
    '1.FSV Mainz 05': 'Mainz 05', 
    'Angers SCO': 'Angers', 
    'Parma Calcio 1913':'Parma', 
    'US Sassuolo': 'Sassuolo', 
    'AJ Auxerre':  'Auxerre', 
    'Sunderland AFC':  'Sunderland', 
    'CA Osasuna': 'Osasuna', 
    'Stade Brestois 29': 'Brest', 
    'SC Freiburg': 'Freiburg', 
    'AC Milan': 'Milan', 
    'US Cremonese': 'Cremonese', 
    'Atalanta BC':  'Atalanta'
}


def get_transfermarkt_transfers(driver, league_name: str, season: int):

    league = NAME_LEAGUE_TRANSFERMARKT[league_name]

    url = (
        f"https://www.transfermarkt.com/{league['slug']}/transfers/"
        f"wettbewerb/{league['code']}/plus/"
        f"?saison_id={season}&s_w=&leihe=1&intern=0&intern=1"
    )

    options = Options()

    # Si quieres ver el navegador, deja esta línea comentada
    options.add_argument("--headless=new")

    options.add_argument("--start-maximized")


    driver = webdriver.Chrome(options=options)
    driver.get(url)

    time.sleep(5)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    transfers = []
    summary = []

    for box in soup.select("div.box"):

        # ==========================
        # Equipo
        # ==========================

        team = box.select_one("h2.content-box-headline a:last-of-type")

        if team is None:
            continue

        team = team.text.strip()

        team_logo = None

        logo = box.select_one("h2.content-box-headline img")

        if logo:
            team_logo = logo.get("src")

        # ==========================
        # Tablas In / Out
        # ==========================

        for table in box.find_all("table"):

            th = table.find("th")

            if th is None:
                continue

            header = th.get_text(strip=True)

            if header == "In":
                transfer_type = "In"

            elif header == "Out":
                transfer_type = "Out"

            else:
                continue

            tbody = table.find("tbody")

            if tbody is None:
                continue

            for row in tbody.find_all("tr"):

                cols = row.find_all("td")

                if len(cols) < 9:
                    continue

                # jugador
                player = cols[0].find("a")
                player = player["title"] if player else None

                # edad
                age = cols[1].get_text(strip=True)

                # nacionalidad (solo primera)
                flag = cols[2].find("img")

                nationality = None
                nationality_flag = None

                if flag:
                    nationality = flag.get("title")
                    nationality_flag = flag.get("src")

                # posición
                position = cols[3].get_text(strip=True)

                # posición corta
                short_position = cols[4].get_text(strip=True)

                # valor mercado
                market_value = cols[5].get_text(strip=True)

                # club
                club_logo = None

                img = cols[6].find("img")

                if img:
                    club_logo = img.get("src")

                club = cols[7].find("a")
                club = club.get("title") if club else None

                # fee
                fee = cols[8].get_text(" ", strip=True)

                transfers.append({
                    "league": league_name,
                    "season": season,
                    "team": team,
                    "team_logo": team_logo,
                    "type": transfer_type,
                    "player": player,
                    "age": age,
                    "nationality": nationality,
                    "nationality_flag": nationality_flag,
                    "position": position,
                    "position_short": short_position,
                    "market_value": market_value,
                    "club_left_joined": club,
                    "club_left_joined_logo": club_logo,
                    "fee": fee
                })

        # ==========================
        # Resumen
        # ==========================

        info_boxes = box.select(".transfer-zusatzinfo-box")

        avg_age_in = None
        value_in = None
        expenditure = None

        avg_age_out = None
        value_out = None
        income = None

        if len(info_boxes) >= 1:

            spans = info_boxes[0].find_all("span")

            if len(spans) >= 3:
                avg_age_in = spans[0].text.split(":")[-1].strip()
                value_in = spans[1].text.split(":")[-1].strip()
                expenditure = spans[2].text.split(":")[-1].strip()

        if len(info_boxes) >= 2:

            spans = info_boxes[1].find_all("span")

            if len(spans) >= 3:
                avg_age_out = spans[0].text.split(":")[-1].strip()
                value_out = spans[1].text.split(":")[-1].strip()
                income = spans[2].text.split(":")[-1].strip()

        transfer_record = box.select_one(".table-footer span")

        summary.append({
            "league": league_name,
            "season": season,
            "team": team,
            "team_logo": team_logo,
            "avg_age_in": avg_age_in,
            "market_value_in": value_in,
            "expenditure": expenditure,
            "avg_age_out": avg_age_out,
            "market_value_out": value_out,
            "income": income,
            "transfer_record": transfer_record.get_text(strip=True) if transfer_record else None
        })

    return pd.DataFrame(transfers), pd.DataFrame(summary)


def get_transfermarkt_transfers_all(season):

    options = Options()

    # Descomenta si no quieres ver el navegador
    options.add_argument("--headless=new")

    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)

    transfers_all = []
    summary_all = []

    try:

        for league in NAME_LEAGUE_TRANSFERMARKT:

            print(f"Scraping {league}...")

            df_transfers, df_summary = get_transfermarkt_transfers(
                driver=driver,
                league_name=league,
                season=season
            )

            transfers_all.append(df_transfers)
            summary_all.append(df_summary)

            time.sleep(2)

    finally:
        driver.quit()

    df_transfers_all = pd.concat(transfers_all, ignore_index=True)
    df_transfers_all['team'] = df_transfers_all['team'].replace(MAPPING_TEAM_NAME_TRANSFERMARKET_TO_FOTMOB)
    df_summary_all = pd.concat(summary_all, ignore_index=True)
    df_summary_all['team'] = df_summary_all['team'].replace(MAPPING_TEAM_NAME_TRANSFERMARKET_TO_FOTMOB)

    return df_transfers_all, df_summary_all

In [272]:
df_transfers, df_summary = get_transfermarkt_transfers_all('2025')

Scraping LaLiga...
Scraping Bundesliga...
Scraping Ligue 1...
Scraping Serie A...
Scraping Premier League...


In [274]:
df_transfers

,league,season,team,team_logo,type,player,age,nationality,nationality_flag,position,position_short,market_value,club_left_joined,club_left_joined_logo,fee
0,LaLiga,2025,Barcelona,https://tmssl.akamaized.net//images/wappen/hom...,In,Joan García,24,Spain,https://tmssl.akamaized.net//images/flagge/tin...,Goalkeeper,GK,€25.00m,RCD Espanyol Barcelona,https://tmssl.akamaized.net//images/wappen/tin...,€25.00m
1,LaLiga,2025,Barcelona,https://tmssl.akamaized.net//images/wappen/hom...,In,Roony Bardghji,19,Sweden,https://tmssl.akamaized.net//images/flagge/tin...,Right Winger,RW,€7.50m,FC Copenhagen,https://tmssl.akamaized.net//images/wappen/tin...,€2.50m
2,LaLiga,2025,Barcelona,https://tmssl.akamaized.net//images/wappen/hom...,In,João Cancelo,31,Portugal,https://tmssl.akamaized.net//images/flagge/tin...,Right-Back,RB,€10.00m,Al-Hilal SFC,https://tmssl.akamaized.net//images/wappen/tin...,loan transfer
3,LaLiga,2025,Barcelona,https://tmssl.akamaized.net//images/wappen/hom...,In,Marcus Rashford,27,England,https://tmssl.akamaized.net//images/flagge/tin...,Left Winger,LW,€50.00m,Manchester United,https://tmssl.akamaized.net//images/wappen/tin...,loan transfer
4,LaLiga,2025,Barcelona,https://tmssl.akamaized.net//images/wappen/hom...,In,Áron Yaakobishvili,19,Hungary,https://tmssl.akamaized.net//images/flagge/tin...,Goalkeeper,GK,€300k,FC Barcelona Atlètic,https://tmssl.akamaized.net//images/wappen/tin...,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4495,Premier League,2025,Wolverhampton Wanderers,https://tmssl.akamaized.net//images/wappen/hom...,Out,Pedro Lima,19,Brazil,https://tmssl.akamaized.net//images/flagge/tin...,Right-Back,RB,€5.00m,FC Porto B,https://tmssl.akamaized.net//images/wappen/tin...,loan transfer
4496,Premier League,2025,Wolverhampton Wanderers,https://tmssl.akamaized.net//images/wappen/hom...,Out,Ki-Jana Hoever,23,Netherlands,https://tmssl.akamaized.net//images/flagge/tin...,Right-Back,RB,€5.00m,Sheffield United,https://tmssl.akamaized.net//images/wappen/tin...,loan transfer
4497,Premier League,2025,Wolverhampton Wanderers,https://tmssl.akamaized.net//images/wappen/hom...,Out,Nasser Djiga,22,Burkina Faso,https://tmssl.akamaized.net//images/flagge/tin...,Centre-Back,CB,€10.00m,Rangers FC,https://tmssl.akamaized.net//images/wappen/tin...,loan transfer
4498,Premier League,2025,Wolverhampton Wanderers,https://tmssl.akamaized.net//images/wappen/hom...,Out,Carlos Forbs,21,Portugal,https://tmssl.akamaized.net//images/flagge/tin...,Right Winger,RW,€8.00m,Ajax Amsterdam,https://tmssl.akamaized.net//images/wappen/tin...,End of loan 30/06/2025


In [256]:
df_summary

,league,season,team,team_logo,avg_age_in,market_value_in,expenditure,avg_age_out,market_value_out,income,transfer_record
0,LaLiga,2025,FC Barcelona,https://tmssl.akamaized.net//images/wappen/hom...,24.2,€113.30m,€27.50m,24.7,€70.80m,€31.20m,€3.70m
1,LaLiga,2025,Atlético de Madrid,https://tmssl.akamaized.net//images/wappen/hom...,25.1,€321.20m,€230.95m,28.4,€201.70m,€145.50m,€-85.45m
2,LaLiga,2025,Real Madrid,https://tmssl.akamaized.net//images/wappen/hom...,21.0,€213.00m,€167.50m,25.8,€39.50m,€27.00m,€-140.50m
3,LaLiga,2025,Athletic Bilbao,https://tmssl.akamaized.net//images/wappen/hom...,23.2,€47.80m,€22.00m,25.0,€32.05m,€4.20m,€-17.80m
4,LaLiga,2025,Villarreal CF,https://tmssl.akamaized.net//images/wappen/hom...,23.6,€155.10m,€105.50m,25.1,€160.60m,€108.00m,€2.50m
...,...,...,...,...,...,...,...,...,...,...,...
91,Premier League,2025,Nottingham Forest,https://tmssl.akamaized.net//images/wappen/hom...,24.1,€305.90m,€241.83m,25.8,€229.05m,€130.40m,€-111.43m
92,Premier League,2025,Sunderland AFC,https://tmssl.akamaized.net//images/wappen/hom...,23.7,€217.03m,€213.44m,24.0,€137.28m,€57.10m,€-156.34m
93,Premier League,2025,Tottenham Hotspur,https://tmssl.akamaized.net//images/wappen/hom...,22.8,€368.50m,€265.60m,24.3,€237.00m,€82.70m,€-182.90m
94,Premier League,2025,West Ham United,https://tmssl.akamaized.net//images/wappen/hom...,25.3,€224.10m,€208.35m,28.2,€286.30m,€147.00m,€-61.35m
